In [ ]:
# Step 1: Install required libraries
%pip install -q pandas pyarrow huggingface_hub

In [ ]:
import pandas as pd

# Load the RCAEval case index only
url = "https://huggingface.co/datasets/phamquiluan/RCAEval/resolve/main/cases.parquet"

cases = pd.read_parquet(url)

print("Total cases:", len(cases))

# Select Online Boutique cases from RE1
re1_ob = cases[cases["dataset"] == "RE1-OB"].copy()

print("\nRE1-OB cases:", len(re1_ob))

display(
    re1_ob[
        [
            "case",
            "root_cause_service",
            "fault",
            "repetition",
            "n_metrics",
            "n_timesteps"
        ]
    ].head(15)
)

Total cases: 735

RE1-OB cases: 125


,case,root_cause_service,fault,repetition,n_metrics,n_timesteps
0,re1ob_adservice_cpu_1,adservice,cpu,1,49,4201
1,re1ob_adservice_cpu_2,adservice,cpu,2,49,4201
2,re1ob_adservice_cpu_3,adservice,cpu,3,49,4201
3,re1ob_adservice_cpu_4,adservice,cpu,4,49,4201
4,re1ob_adservice_cpu_5,adservice,cpu,5,49,4201
5,re1ob_adservice_delay_1,adservice,delay,1,57,721
6,re1ob_adservice_delay_2,adservice,delay,2,57,721
7,re1ob_adservice_delay_3,adservice,delay,3,57,721
8,re1ob_adservice_delay_4,adservice,delay,4,57,721
9,re1ob_adservice_delay_5,adservice,delay,5,58,721


In [ ]:
# Step 2: Download one failure case

from huggingface_hub import hf_hub_download

case_name = "re1ob_adservice_cpu_1"

# Download telemetry data for this case
metrics_path = hf_hub_download(
    repo_id="phamquiluan/RCAEval",
    repo_type="dataset",
    filename=f"{case_name}/metrics.parquet"
)

# Download fault injection timestamp
inject_path = hf_hub_download(
    repo_id="phamquiluan/RCAEval",
    repo_type="dataset",
    filename=f"{case_name}/inject_time.txt"
)

# Load telemetry into the existing pandas environment
df = pd.read_parquet(metrics_path)

with open(inject_path, "r") as file:
    inject_time = int(file.read().strip())

# Inspect the dataset
print("Case:", case_name)
print("Root cause service:", "adservice")
print("Fault type:", "CPU")
print("Telemetry shape:", df.shape)
print("Fault injection timestamp:", inject_time)

print("\nFirst 5 rows:")
display(df.head())

print("\nColumn names:")
print(df.columns.tolist())

print("\nMissing values:", df.isnull().sum().sum())

print("\nData types:")
print(df.dtypes.value_counts())

print("\nTime range:")
print(df["time"].min(), "to", df["time"].max())

print("\nRows before fault:", (df["time"] < inject_time).sum())
print("Rows after fault:", (df["time"] >= inject_time).sum())

Case: re1ob_adservice_cpu_1
Root cause service: adservice
Fault type: CPU
Telemetry shape: (4201, 50)
Fault injection timestamp: 1685202688

First 5 rows:


,time,adservice_cpu,cartservice_cpu,checkoutservice_cpu,currencyservice_cpu,emailservice_cpu,frontend_cpu,main_cpu,paymentservice_cpu,productcatalogservice_cpu,...,currencyservice_latency,emailservice_latency,frontend_latency,paymentservice_latency,productcatalogservice_latency,recommendationservice_latency,shippingservice_latency,PassthroughCluster_error,frontend_error,frontend-external_error
0,1685200588,44.503283,4.965416,1.947450,7.952500,0.992255,17.725886,7.911334,0.457972,11.843233,...,0.004957,0.0046,0.381486,0.004631,0.006510,0.014185,0.004676,0.0,0.0,0.0
1,1685200589,37.376351,4.957174,1.947450,7.952500,0.992255,17.707375,7.911334,0.408782,11.903299,...,0.004958,0.0046,0.399897,0.004633,0.006516,0.014185,0.004678,0.0,0.0,0.0
2,1685200590,37.376351,4.957174,1.681948,7.836973,0.992255,17.657918,7.911334,0.408782,11.903299,...,0.004958,0.0046,0.399897,0.004631,0.006513,0.014185,0.004677,0.0,0.0,0.0
3,1685200591,37.376351,4.957174,1.649491,7.836973,0.992255,17.657918,7.911334,0.458569,11.903299,...,0.004989,0.0046,0.399897,0.004631,0.006742,0.015335,0.004668,0.0,0.0,0.0
4,1685200592,37.376351,4.957174,1.617033,7.836973,0.992255,17.657918,7.911334,0.458569,11.903299,...,0.005240,0.0046,0.399897,0.004631,0.007011,0.017648,0.004665,0.0,0.0,0.0



Column names:
['time', 'adservice_cpu', 'cartservice_cpu', 'checkoutservice_cpu', 'currencyservice_cpu', 'emailservice_cpu', 'frontend_cpu', 'main_cpu', 'paymentservice_cpu', 'productcatalogservice_cpu', 'recommendationservice_cpu', 'redis_cpu', 'shippingservice_cpu', 'adservice_mem', 'cartservice_mem', 'checkoutservice_mem', 'currencyservice_mem', 'emailservice_mem', 'frontend_mem', 'main_mem', 'paymentservice_mem', 'productcatalogservice_mem', 'recommendationservice_mem', 'redis_mem', 'shippingservice_mem', 'PassthroughCluster_load', 'adservice_load', 'cartservice_load', 'checkoutservice_load', 'currencyservice_load', 'emailservice_load', 'frontend_load', 'frontend-external_load', 'paymentservice_load', 'productcatalogservice_load', 'recommendationservice_load', 'shippingservice_load', 'adservice_latency', 'cartservice_latency', 'checkoutservice_latency', 'currencyservice_latency', 'emailservice_latency', 'frontend_latency', 'paymentservice_latency', 'productcatalogservice_latency',

In [ ]:
import numpy as np
# Step 3: Compare telemetry before and after the fault

metric_columns = [col for col in df.columns if col != "time"]

before = df[df["time"] < inject_time]
after = df[df["time"] >= inject_time]

comparison = pd.DataFrame({
    "before_mean": before[metric_columns].mean(),
    "after_mean": after[metric_columns].mean(),
})

comparison["absolute_change"] = (
    comparison["after_mean"] - comparison["before_mean"]
)

comparison["percentage_change"] = (
    comparison["absolute_change"] /
    comparison["before_mean"].replace(0, np.nan)
) * 100

comparison = comparison.sort_values(
    "percentage_change",
    key=lambda x: x.abs(),
    ascending=False
)

display(comparison.head(15))

,before_mean,after_mean,absolute_change,percentage_change
adservice_cpu,1.827248e+00,9.665278e+01,9.482553e+01,5189.526809
frontend-external_error,3.144524e-02,4.759638e-03,-2.668560e-02,-84.863723
cartservice_latency,9.360956e-03,6.897447e-03,-2.463509e-03,-26.316849
adservice_mem,9.439301e+07,1.003284e+08,5.935433e+06,6.288000
frontend_latency,2.489386e-01,2.341065e-01,-1.483208e-02,-5.958128
productcatalogservice_latency,5.139810e-03,4.928960e-03,-2.108497e-04,-4.102287
cartservice_cpu,6.108271e+00,5.905601e+00,-2.026704e-01,-3.317967
main_mem,7.035670e+07,7.237665e+07,2.019958e+06,2.871024
emailservice_load,3.638830e+00,3.711160e+00,7.233088e-02,1.987751
paymentservice_cpu,4.375684e-01,4.289484e-01,-8.620019e-03,-1.969982


In [ ]:
print("Fault injection time:", inject_time)

print("\nTime relative to fault:")
print("First timestamp :", df["time"].min() - inject_time)
print("Last timestamp  :", df["time"].max() - inject_time)

print("\nNumber of timestamps:")
print("Before fault:", len(before))
print("After fault :", len(after))

Fault injection time: 1685202688

Time relative to fault:
First timestamp : -2100
Last timestamp  : 2100

Number of timestamps:
Before fault: 2100
After fault : 2101


In [ ]:
# Check the structure of all RE1-OB cases

print("Number of RE1-OB cases:", len(re1_ob))

print("\nFault types:")
print(re1_ob["fault"].value_counts())

print("\nRoot-cause services:")
print(re1_ob["root_cause_service"].value_counts())

print("\nTimesteps:")
print(re1_ob["n_timesteps"].value_counts())

print("\nMetrics:")
print(re1_ob["n_metrics"].value_counts())

Number of RE1-OB cases: 125

Fault types:
fault
cpu      25
delay    25
disk     25
loss     25
mem      25
Name: count, dtype: int64

Root-cause services:
root_cause_service
adservice                25
cartservice              25
checkoutservice          25
currencyservice          25
productcatalogservice    25
Name: count, dtype: int64

Timesteps:
n_timesteps
721     75
4201    48
2679     1
63       1
Name: count, dtype: int64

Metrics:
n_metrics
49    49
59    25
58    24
57    20
56     7
Name: count, dtype: int64


In [ ]:
# Step 4: Download telemetry + injection time for all RE1-OB cases

from tqdm.auto import tqdm

case_files = {}

for case_name in tqdm(re1_ob["case"], desc="Downloading cases"):

    metrics_path = hf_hub_download(
        repo_id="phamquiluan/RCAEval",
        repo_type="dataset",
        filename=f"{case_name}/metrics.parquet"
    )

    inject_path = hf_hub_download(
        repo_id="phamquiluan/RCAEval",
        repo_type="dataset",
        filename=f"{case_name}/inject_time.txt"
    )

    case_files[case_name] = {
        "metrics": metrics_path,
        "inject_time": inject_path
    }

print("\nDownloaded cases:", len(case_files))

re1ob_adservice_cpu_2/metrics.parquet: reconstructing file:   0%|          |  0.00B /  386kB            

re1ob_adservice_cpu_2/metrics.parquet: downloading bytes:           |  0.00B            

inject_time.txt:   0%|          | 0.00/10.0 [00:00<?, ?B/s]

re1ob_adservice_cpu_3/metrics.parquet: reconstructing file:   0%|          |  0.00B /  339kB            

re1ob_adservice_cpu_3/metrics.parquet: downloading bytes:           |  0.00B            

...

inject_time.txt:   0%|          | 0.00/10.0 [00:00<?, ?B/s]

re1ob_productcatalogservice_mem_5/metric(…): reconstructing file:   0%|          |  0.00B /  408kB            

re1ob_productcatalogservice_mem_5/metric(…): downloading bytes:           |  0.00B            

inject_time.txt:   0%|          | 0.00/10.0 [00:00<?, ?B/s]


Downloaded cases: 125


In [ ]:
print(case_files["re1ob_adservice_cpu_1"])

{'metrics': '/root/.cache/huggingface/hub/datasets--phamquiluan--RCAEval/snapshots/afeacb11bcc94dadfd1c8f483ee4377b2b8b614e/re1ob_adservice_cpu_1/metrics.parquet', 'inject_time': '/root/.cache/huggingface/hub/datasets--phamquiluan--RCAEval/snapshots/afeacb11bcc94dadfd1c8f483ee4377b2b8b614e/re1ob_adservice_cpu_1/inject_time.txt'}


In [ ]:
def extract_features(case_name):
    """
    Convert one RCAEval incident's time-series telemetry
    into a single fixed-size feature vector.
    """

    # Load telemetry
    data = pd.read_parquet(case_files[case_name]["metrics"])

    # Load fault injection time
    with open(case_files[case_name]["inject_time"], "r") as file:
        injection_time = int(file.read().strip())

    # Split telemetry into before and after fault
    before = data[data["time"] < injection_time]
    after = data[data["time"] >= injection_time]

    # All telemetry columns except timestamp
    metrics = [col for col in data.columns if col != "time"]

    features = {}

    for metric in metrics:

        before_values = before[metric]
        after_values = after[metric]

        before_mean = before_values.mean()
        after_mean = after_values.mean()

        # Basic statistics
        features[f"{metric}_before_mean"] = before_mean
        features[f"{metric}_after_mean"] = after_mean

        features[f"{metric}_before_std"] = before_values.std()
        features[f"{metric}_after_std"] = after_values.std()

        features[f"{metric}_after_max"] = after_values.max()

        # Absolute change
        features[f"{metric}_delta"] = after_mean - before_mean

        # Relative change
        denominator = abs(before_mean) + 1e-8
        features[f"{metric}_relative_change"] = (
            (after_mean - before_mean) / denominator
        )

    return features

In [ ]:
test_features = extract_features("re1ob_adservice_cpu_1")

print("Number of features:", len(test_features))

display(
    pd.Series(test_features).head(20)
)

Number of features: 343


,0
adservice_cpu_before_mean,1.827248
adservice_cpu_after_mean,96.652783
adservice_cpu_before_std,2.128108
adservice_cpu_after_std,11.907982
adservice_cpu_after_max,100.020114
adservice_cpu_delta,94.825535
adservice_cpu_relative_change,51.895268
cartservice_cpu_before_mean,6.108271
cartservice_cpu_after_mean,5.905601
cartservice_cpu_before_std,0.773742


In [ ]:
# Step 6: Build the complete ML dataset

feature_rows = []
metadata = []

for _, row in tqdm(re1_ob.iterrows(), total=len(re1_ob), desc="Extracting features"):

    case_name = row["case"]

    features = extract_features(case_name)
    feature_rows.append(features)

    metadata.append({
        "case": case_name,
        "root_cause_service": row["root_cause_service"],
        "fault": row["fault"],
        "repetition": row["repetition"]
    })

# Convert feature dictionaries into a DataFrame
X = pd.DataFrame(feature_rows)

# Metadata / target
metadata_df = pd.DataFrame(metadata)

y = metadata_df["root_cause_service"]

print("Feature matrix shape:", X.shape)
print("Target shape:", y.shape)

print("\nMissing values:")
print("Total:", X.isnull().sum().sum())

print("\nRoot-cause distribution:")
print(y.value_counts())

print("\nFirst 5 feature rows:")
display(X.iloc[:5, :10])

Extracting features:   0%|          | 0/125 [00:00<?, ?it/s]

Feature matrix shape: (125, 665)
Target shape: (125,)

Missing values:
Total: 36059

Root-cause distribution:
root_cause_service
adservice                25
cartservice              25
checkoutservice          25
currencyservice          25
productcatalogservice    25
Name: count, dtype: int64

First 5 feature rows:


,adservice_cpu_before_mean,adservice_cpu_after_mean,adservice_cpu_before_std,adservice_cpu_after_std,adservice_cpu_after_max,adservice_cpu_delta,adservice_cpu_relative_change,cartservice_cpu_before_mean,cartservice_cpu_after_mean,cartservice_cpu_before_std
0,1.827248,96.652783,2.128108,11.907982,100.020114,94.825535,51.895268,6.108271,5.905601,0.773742
1,1.607083,96.724094,0.108424,11.893947,100.035967,95.117011,59.186105,6.069960,5.895905,0.830728
2,1.624496,96.752463,0.106534,11.510336,100.022958,95.127967,58.558463,6.087101,5.856388,0.796827
3,2.670560,93.942132,0.681886,11.709102,100.013126,91.271572,34.176938,6.091683,5.718182,0.722150
4,2.572038,93.462218,0.704495,11.289499,100.012565,90.890180,35.337811,6.442865,5.482797,1.574694


In [ ]:
# Step 7: Understand why features are missing

missing_counts = X.isnull().sum()

missing_summary = (
    missing_counts[missing_counts > 0]
    .sort_values(ascending=False)
)

print("Total feature columns:", X.shape[1])
print("Columns with missing values:", len(missing_summary))

print("\nTop missing feature columns:")
display(missing_summary.head(30))

Total feature columns: 665
Columns with missing values: 665

Top missing feature columns:


,0
carts_error_after_mean,125
orders_error_relative_change,125
orders_error_delta,125
orders_error_after_max,125
orders_error_after_std,125
InboundPassthroughClusterIpv4_error_delta,125
InboundPassthroughClusterIpv4_error_relative_change,125
InboundPassthroughClusterIpv4_error_after_max,125
carts_load_delta,125
carts_load_after_std,125


In [ ]:
# How many features are complete across ALL 125 cases?

complete_features = X.columns[X.notnull().all()]

print("Features available in all 125 cases:", len(complete_features))
print("Features missing in at least one case:", len(X.columns) - len(complete_features))

Features available in all 125 cases: 0
Features missing in at least one case: 665


In [ ]:
# Step 8: Understand metric availability across fault types

metric_sets = {}

for _, row in tqdm(
    re1_ob.iterrows(),
    total=len(re1_ob),
    desc="Checking metric availability"
):
    case_name = row["case"]

    data = pd.read_parquet(
        case_files[case_name]["metrics"]
    )

    metric_sets[case_name] = set(
        col for col in data.columns if col != "time"
    )

# Metrics common to ALL cases
common_all = set.intersection(*metric_sets.values())

print("Metrics common to ALL 125 cases:", len(common_all))

# Metrics common within each fault type
for fault in sorted(re1_ob["fault"].unique()):

    fault_cases = re1_ob[
        re1_ob["fault"] == fault
    ]["case"]

    fault_sets = [
        metric_sets[case]
        for case in fault_cases
    ]

    common_fault = set.intersection(*fault_sets)

    print(
        f"{fault:>6} -> "
        f"{len(common_fault)} common metrics"
    )

Checking metric availability:   0%|          | 0/125 [00:00<?, ?it/s]

Metrics common to ALL 125 cases: 24
   cpu -> 49 common metrics
 delay -> 55 common metrics
  disk -> 55 common metrics
  loss -> 57 common metrics
   mem -> 49 common metrics


In [ ]:
# Step 9: See the 24 metrics available in every case

print("Common metrics available in all 125 cases:\n")

for i, metric in enumerate(sorted(common_all), start=1):
    print(f"{i:2}. {metric}")

print("\nTotal common metrics:", len(common_all))

Common metrics available in all 125 cases:

 1. adservice_cpu
 2. adservice_mem
 3. cartservice_cpu
 4. cartservice_mem
 5. checkoutservice_cpu
 6. checkoutservice_mem
 7. currencyservice_cpu
 8. currencyservice_mem
 9. emailservice_cpu
10. emailservice_mem
11. frontend_cpu
12. frontend_mem
13. main_cpu
14. main_mem
15. paymentservice_cpu
16. paymentservice_mem
17. productcatalogservice_cpu
18. productcatalogservice_mem
19. recommendationservice_cpu
20. recommendationservice_mem
21. redis_cpu
22. redis_mem
23. shippingservice_cpu
24. shippingservice_mem

Total common metrics: 24


In [ ]:
# Step 10: Feature extraction using only universally available metrics

common_metrics = sorted(common_all)

def extract_common_features(case_name):
    """
    Extract fixed-size features using only metrics
    available in every RE1-OB case.
    """

    data = pd.read_parquet(
        case_files[case_name]["metrics"]
    )

    with open(case_files[case_name]["inject_time"], "r") as file:
        injection_time = int(file.read().strip())

    before = data[data["time"] < injection_time]
    after = data[data["time"] >= injection_time]

    features = {}

    for metric in common_metrics:

        before_values = before[metric]
        after_values = after[metric]

        before_mean = before_values.mean()
        after_mean = after_values.mean()

        features[f"{metric}_before_mean"] = before_mean
        features[f"{metric}_after_mean"] = after_mean

        features[f"{metric}_before_std"] = before_values.std()
        features[f"{metric}_after_std"] = after_values.std()

        features[f"{metric}_after_max"] = after_values.max()

        features[f"{metric}_delta"] = (
            after_mean - before_mean
        )

        features[f"{metric}_relative_change"] = (
            (after_mean - before_mean)
            / (abs(before_mean) + 1e-8)
        )

    return features

In [ ]:
test_features = extract_common_features(
    "re1ob_adservice_cpu_1"
)

print("Number of features:", len(test_features))

display(
    pd.Series(test_features).head(20)
)

Number of features: 168


,0
adservice_cpu_before_mean,1.827248e+00
adservice_cpu_after_mean,9.665278e+01
adservice_cpu_before_std,2.128108e+00
adservice_cpu_after_std,1.190798e+01
adservice_cpu_after_max,1.000201e+02
adservice_cpu_delta,9.482553e+01
adservice_cpu_relative_change,5.189527e+01
adservice_mem_before_mean,9.439301e+07
adservice_mem_after_mean,1.003284e+08
adservice_mem_before_std,6.654695e+05


In [ ]:
# Step 11: Build the clean V1 ML dataset

feature_rows = []
labels = []
case_names = []

for _, row in tqdm(
    re1_ob.iterrows(),
    total=len(re1_ob),
    desc="Building ML dataset"
):
    case_name = row["case"]

    features = extract_common_features(case_name)

    feature_rows.append(features)
    labels.append(row["root_cause_service"])
    case_names.append(case_name)

# Feature matrix
X = pd.DataFrame(feature_rows)

# Target
y = pd.Series(
    labels,
    name="root_cause_service"
)

# Case identifiers (kept separately, NOT used as model features)
case_ids = pd.Series(
    case_names,
    name="case"
)

print("X shape:", X.shape)
print("y shape:", y.shape)

print("\nMissing values:", X.isnull().sum().sum())

print("\nTarget distribution:")
print(y.value_counts())

print("\nFeature data types:")
print(X.dtypes.value_counts())

Building ML dataset:   0%|          | 0/125 [00:00<?, ?it/s]

X shape: (125, 168)
y shape: (125,)

Missing values: 216

Target distribution:
root_cause_service
adservice                25
cartservice              25
checkoutservice          25
currencyservice          25
productcatalogservice    25
Name: count, dtype: int64

Feature data types:
float64    168
Name: count, dtype: int64


In [ ]:
# Step 12: Investigate missing values

missing_by_feature = X.isnull().sum()

print("Total missing values:", missing_by_feature.sum())
print("Features containing NaN:", (missing_by_feature > 0).sum())

print("\nMissing values by feature:")
display(
    missing_by_feature[missing_by_feature > 0]
    .sort_values(ascending=False)
)

Total missing values: 216
Features containing NaN: 168

Missing values by feature:


,0
adservice_cpu_relative_change,2
adservice_cpu_delta,2
checkoutservice_cpu_relative_change,2
checkoutservice_cpu_delta,2
cartservice_cpu_delta,2
...,...
shippingservice_mem_before_mean,1
shippingservice_mem_before_std,1
shippingservice_mem_after_mean,1
shippingservice_mem_after_max,1


In [ ]:
missing_cases = X.isnull().sum(axis=1)

print("Cases containing NaN:", (missing_cases > 0).sum())
print("Maximum missing features in one case:", missing_cases.max())

print("\nMissing features per affected case:")
display(
    pd.DataFrame({
        "case": case_ids,
        "root_cause": y,
        "missing_features": missing_cases
    })[missing_cases > 0]
)

Cases containing NaN: 2
Maximum missing features in one case: 120

Missing features per affected case:


,case,root_cause,missing_features
90,re1ob_currencyservice_loss_1,currencyservice,96
102,re1ob_productcatalogservice_cpu_3,productcatalogservice,120


In [ ]:
# Step 13: Inspect the two problematic cases

problem_cases = [
    "re1ob_currencyservice_loss_1",
    "re1ob_productcatalogservice_cpu_3"
]

for case_name in problem_cases:

    data = pd.read_parquet(
        case_files[case_name]["metrics"]
    )

    with open(case_files[case_name]["inject_time"], "r") as file:
        injection_time = int(file.read().strip())

    before = data[data["time"] < injection_time]
    after = data[data["time"] >= injection_time]

    print("=" * 70)
    print("Case:", case_name)
    print("Shape:", data.shape)
    print("Injection time:", injection_time)
    print("Before rows:", len(before))
    print("After rows:", len(after))
    print("Raw NaN values:", data.isnull().sum().sum())

    print("\nNaN counts by metric:")
    nan_counts = data.drop(columns=["time"]).isnull().sum()
    display(
        nan_counts[nan_counts > 0]
        .sort_values(ascending=False)
    )

Case: re1ob_currencyservice_loss_1
Shape: (721, 60)
Injection time: 16933142
Before rows: 0
After rows: 721
Raw NaN values: 0

NaN counts by metric:


,0


Case: re1ob_productcatalogservice_cpu_3
Shape: (63, 59)
Injection time: 1685373255
Before rows: 63
After rows: 0
Raw NaN values: 0

NaN counts by metric:


,0


In [ ]:
# Remove cases where the injection timestamp does not
# actually split the telemetry into before and after periods.

valid_cases = []

for _, row in re1_ob.iterrows():
    case_name = row["case"]

    data = pd.read_parquet(case_files[case_name]["metrics"])

    with open(case_files[case_name]["inject_time"], "r") as file:
        injection_time = int(file.read().strip())

    before = data[data["time"] < injection_time]
    after = data[data["time"] >= injection_time]

    if len(before) > 0 and len(after) > 0:
        valid_cases.append(case_name)

print("Total RE1-OB cases:", len(re1_ob))
print("Usable cases:", len(valid_cases))
print("Excluded cases:", len(re1_ob) - len(valid_cases))

print("\nExcluded cases:")
print(set(re1_ob["case"]) - set(valid_cases))

Total RE1-OB cases: 125
Usable cases: 123
Excluded cases: 2

Excluded cases:
{'re1ob_productcatalogservice_cpu_3', 're1ob_currencyservice_loss_1'}


In [ ]:
feature_rows = []
labels = []
case_names = []

for case_name in tqdm(valid_cases, desc="Building clean ML dataset"):
    features = extract_common_features(case_name)

    feature_rows.append(features)

    # Get the metadata for this case
    row = re1_ob[re1_ob["case"] == case_name].iloc[0]

    labels.append(row["root_cause_service"])
    case_names.append(case_name)

X = pd.DataFrame(feature_rows)
y = pd.Series(labels, name="root_cause_service")
case_ids = pd.Series(case_names, name="case")

print("X shape:", X.shape)
print("y shape:", y.shape)

print("\nMissing values:", X.isnull().sum().sum())

print("\nTarget distribution:")
print(y.value_counts().sort_index())

Building clean ML dataset:   0%|          | 0/123 [00:00<?, ?it/s]

X shape: (123, 168)
y shape: (123,)

Missing values: 0

Target distribution:
root_cause_service
adservice                25
cartservice              25
checkoutservice          25
currencyservice          24
productcatalogservice    24
Name: count, dtype: int64


In [ ]:
metadata = re1_ob[
    re1_ob["case"].isin(valid_cases)
][
    ["case", "root_cause_service", "fault", "repetition"]
].copy()

# Each service + fault combination is one experimental group.
metadata["group"] = (
    metadata["root_cause_service"] + "_" + metadata["fault"]
)

print("Number of groups:", metadata["group"].nunique())

print("\nCases per group:")
print(metadata.groupby("group").size().value_counts().sort_index())

print("\nExample groups:")
display(
    metadata.sort_values(
        ["root_cause_service", "fault", "repetition"]
    ).head(20)
)

Number of groups: 25

Cases per group:
4     2
5    23
Name: count, dtype: int64

Example groups:


,case,root_cause_service,fault,repetition,group
0,re1ob_adservice_cpu_1,adservice,cpu,1,adservice_cpu
1,re1ob_adservice_cpu_2,adservice,cpu,2,adservice_cpu
2,re1ob_adservice_cpu_3,adservice,cpu,3,adservice_cpu
3,re1ob_adservice_cpu_4,adservice,cpu,4,adservice_cpu
4,re1ob_adservice_cpu_5,adservice,cpu,5,adservice_cpu
5,re1ob_adservice_delay_1,adservice,delay,1,adservice_delay
6,re1ob_adservice_delay_2,adservice,delay,2,adservice_delay
7,re1ob_adservice_delay_3,adservice,delay,3,adservice_delay
8,re1ob_adservice_delay_4,adservice,delay,4,adservice_delay
9,re1ob_adservice_delay_5,adservice,delay,5,adservice_delay


In [ ]:
from sklearn.model_selection import StratifiedGroupKFold

# Align metadata with X/y
metadata = metadata.set_index("case").loc[case_ids].reset_index()

groups = metadata["group"]

cv = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

print("Number of folds:", cv.get_n_splits())
print("Number of groups:", groups.nunique())

Number of folds: 5
Number of groups: 25


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

logistic_model = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(
        max_iter=2000,
        C=1.0,
        random_state=42
    ))
])

scores = cross_val_score(
    logistic_model,
    X,
    y,
    cv=cv,
    groups=groups,
    scoring="accuracy"
)

print("Fold accuracies:")
for i, score in enumerate(scores, start=1):
    print(f"Fold {i}: {score:.4f}")

print(f"\nMean accuracy: {scores.mean():.4f}")
print(f"Std deviation: {scores.std():.4f}")

Fold accuracies:
Fold 1: 0.4000
Fold 2: 0.6800
Fold 3: 0.4000
Fold 4: 0.4167
Fold 5: 0.6667

Mean accuracy: 0.5127
Std deviation: 0.1314


In [ ]:
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import confusion_matrix, classification_report

y_pred = cross_val_predict(
    logistic_model,
    X,
    y,
    cv=cv,
    groups=groups
)

print("Classification Report:\n")
print(
    classification_report(
        y,
        y_pred,
        digits=3
    )
)

cm = confusion_matrix(
    y,
    y_pred,
    labels=sorted(y.unique())
)

print("\nConfusion Matrix:")
display(
    pd.DataFrame(
        cm,
        index=sorted(y.unique()),
        columns=sorted(y.unique())
    )
)

Classification Report:

                       precision    recall  f1-score   support

            adservice      1.000     0.800     0.889        25
          cartservice      0.433     0.520     0.473        25
      checkoutservice      0.786     0.440     0.564        25
      currencyservice      0.227     0.417     0.294        24
productcatalogservice      0.600     0.375     0.462        24

             accuracy                          0.512       123
            macro avg      0.609     0.510     0.536       123
         weighted avg      0.612     0.512     0.539       123


Confusion Matrix:


,adservice,cartservice,checkoutservice,currencyservice,productcatalogservice
adservice,20,0,1,4,0
cartservice,0,13,2,10,0
checkoutservice,0,9,11,5,0
currencyservice,0,8,0,10,6
productcatalogservice,0,0,0,15,9


In [ ]:
from sklearn.ensemble import ExtraTreesClassifier

extra_trees = ExtraTreesClassifier(
    n_estimators=300,
    max_features="sqrt",
    min_samples_leaf=2,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

extra_scores = cross_val_score(
    extra_trees,
    X,
    y,
    cv=cv,
    groups=groups,
    scoring="accuracy"
)

print("ExtraTrees fold accuracies:")

for i, score in enumerate(extra_scores, start=1):
    print(f"Fold {i}: {score:.4f}")

print(f"\nMean accuracy: {extra_scores.mean():.4f}")
print(f"Std deviation: {extra_scores.std():.4f}")

ExtraTrees fold accuracies:
Fold 1: 0.4000
Fold 2: 0.6000
Fold 3: 0.5200
Fold 4: 0.4167
Fold 5: 0.5833

Mean accuracy: 0.5040
Std deviation: 0.0827


In [ ]:
from collections import Counter

metric_counter = Counter()

for case_name in valid_cases:
    data = pd.read_parquet(case_files[case_name]["metrics"])

    for metric in data.columns:
        if metric != "time":
            metric_counter[metric] += 1

metric_frequency = (
    pd.DataFrame(
        metric_counter.items(),
        columns=["metric", "case_count"]
    )
    .sort_values("case_count", ascending=False)
)

print("Total unique metrics:", len(metric_frequency))

display(metric_frequency.head(40))

Total unique metrics: 86


,metric,case_count
0,adservice_cpu,123
1,cartservice_cpu,123
2,checkoutservice_cpu,123
3,currencyservice_cpu,123
4,emailservice_cpu,123
5,frontend_cpu,123
6,main_cpu,123
7,paymentservice_cpu,123
8,productcatalogservice_cpu,123
9,recommendationservice_cpu,123


In [ ]:
print("Metric availability distribution:\n")

print(
    metric_frequency["case_count"]
    .value_counts()
    .sort_index(ascending=False)
)

Metric availability distribution:

case_count
123    24
114     1
108     1
74     31
56      1
49     23
19      1
5       2
2       1
1       1
Name: count, dtype: int64


In [ ]:
coverage_threshold = 56

selected_metrics = metric_frequency[
    metric_frequency["case_count"] >= coverage_threshold
]["metric"].tolist()

print("Selected metrics:", len(selected_metrics))
print()

for metric in selected_metrics:
    count = metric_frequency.loc[
        metric_frequency["metric"] == metric,
        "case_count"
    ].iloc[0]

    print(f"{metric:<45} {count}/123")

Selected metrics: 58

adservice_cpu                                 123/123
cartservice_cpu                               123/123
checkoutservice_cpu                           123/123
currencyservice_cpu                           123/123
emailservice_cpu                              123/123
frontend_cpu                                  123/123
main_cpu                                      123/123
paymentservice_cpu                            123/123
productcatalogservice_cpu                     123/123
recommendationservice_cpu                     123/123
redis_cpu                                     123/123
shippingservice_cpu                           123/123
adservice_mem                                 123/123
cartservice_mem                               123/123
checkoutservice_mem                           123/123
currencyservice_mem                           123/123
emailservice_mem                              123/123
frontend_mem                                  123/123
main_m

In [ ]:
def metric_type(metric):
    if "_cpu" in metric:
        return "CPU"
    elif "_mem" in metric:
        return "Memory"
    elif "latency" in metric:
        return "Latency"
    elif "error" in metric:
        return "Error"
    elif "workload" in metric:
        return "Workload"
    else:
        return "Other"


selected_metric_df = metric_frequency[
    metric_frequency["metric"].isin(selected_metrics)
].copy()

selected_metric_df["type"] = selected_metric_df["metric"].apply(metric_type)

print("Metrics by type:")
print(
    selected_metric_df["type"]
    .value_counts()
    .sort_index()
)

print("\nDetailed breakdown:")
display(
    selected_metric_df[
        ["metric", "type", "case_count"]
    ].sort_values(["type", "metric"])
)

Metrics by type:
type
CPU         12
Error        3
Latency     20
Memory      12
Workload    11
Name: count, dtype: int64

Detailed breakdown:


,metric,type,case_count
0,adservice_cpu,CPU,123
1,cartservice_cpu,CPU,123
2,checkoutservice_cpu,CPU,123
3,currencyservice_cpu,CPU,123
4,emailservice_cpu,CPU,123
5,frontend_cpu,CPU,123
6,main_cpu,CPU,123
7,paymentservice_cpu,CPU,123
8,productcatalogservice_cpu,CPU,123
9,recommendationservice_cpu,CPU,123


In [ ]:
def extract_temporal_features(case_name):
    """
    Extract RCA-oriented temporal features from selected telemetry.

    Post-injection telemetry is divided into three windows:
    early, middle, and late.
    """

    data = pd.read_parquet(case_files[case_name]["metrics"])

    with open(case_files[case_name]["inject_time"], "r") as file:
        injection_time = int(file.read().strip())

    before = data[data["time"] < injection_time]
    after = data[data["time"] >= injection_time]

    features = {}

    for metric in selected_metrics:

        # If this metric does not exist in this case,
        # leave its features as NaN.
        if metric not in data.columns:
            for suffix in [
                "before_mean",
                "before_std",
                "early_mean",
                "middle_mean",
                "late_mean",
                "delta",
                "relative_change"
            ]:
                features[f"{metric}_{suffix}"] = float("nan")

            continue

        before_values = before[metric].dropna()
        after_values = after[metric].dropna()

        # Split the post-injection period into 3 chronological windows
        n = len(after_values)

        if n == 0:
            early = middle = late = pd.Series(dtype=float)
        else:
            split1 = n // 3
            split2 = 2 * n // 3

            early = after_values.iloc[:split1]
            middle = after_values.iloc[split1:split2]
            late = after_values.iloc[split2:]

        before_mean = before_values.mean()

        early_mean = early.mean()
        middle_mean = middle.mean()
        late_mean = late.mean()

        # Overall post-incident change
        after_mean = after_values.mean()

        features[f"{metric}_before_mean"] = before_mean
        features[f"{metric}_before_std"] = before_values.std()

        features[f"{metric}_early_mean"] = early_mean
        features[f"{metric}_middle_mean"] = middle_mean
        features[f"{metric}_late_mean"] = late_mean

        features[f"{metric}_delta"] = (
            after_mean - before_mean
        )

        features[f"{metric}_relative_change"] = (
            (after_mean - before_mean)
            / (abs(before_mean) + 1e-8)
        )

    return features

In [ ]:
test_case = valid_cases[0]

test_features = extract_temporal_features(test_case)

print("Test case:", test_case)
print("Number of features:", len(test_features))

display(
    pd.DataFrame(
        list(test_features.items()),
        columns=["feature", "value"]
    ).head(20)
)

Test case: re1ob_adservice_cpu_1
Number of features: 406


,feature,value
0,adservice_cpu_before_mean,1.827248
1,adservice_cpu_before_std,2.128108
2,adservice_cpu_early_mean,93.764326
3,adservice_cpu_middle_mean,98.213711
4,adservice_cpu_late_mean,97.978419
5,adservice_cpu_delta,94.825535
6,adservice_cpu_relative_change,51.895268
7,cartservice_cpu_before_mean,6.108271
8,cartservice_cpu_before_std,0.773742
9,cartservice_cpu_early_mean,6.170311


In [ ]:
temporal_rows = []
temporal_labels = []
temporal_case_names = []

for case_name in tqdm(
    valid_cases,
    desc="Building temporal ML dataset"
):
    features = extract_temporal_features(case_name)

    temporal_rows.append(features)

    row = re1_ob[
        re1_ob["case"] == case_name
    ].iloc[0]

    temporal_labels.append(
        row["root_cause_service"]
    )

    temporal_case_names.append(case_name)

X_temporal = pd.DataFrame(temporal_rows)

y_temporal = pd.Series(
    temporal_labels,
    name="root_cause_service"
)

temporal_case_ids = pd.Series(
    temporal_case_names,
    name="case"
)

print("X_temporal shape:", X_temporal.shape)
print("y_temporal shape:", y_temporal.shape)

print(
    "\nTotal missing values:",
    X_temporal.isnull().sum().sum()
)

print("\nMissing values by case:")
print(
    X_temporal.isnull().sum(axis=1)
    .value_counts()
    .sort_index()
)

Building temporal ML dataset:   0%|          | 0/123 [00:00<?, ?it/s]

X_temporal shape: (123, 406)
y_temporal shape: (123,)

Total missing values: 11278

Missing values by case:
0      37
4       2
7      28
14      7
224    49
Name: count, dtype: int64


In [ ]:
missing_by_case = X_temporal.isnull().sum(axis=1)

missing_summary = pd.DataFrame({
    "root_cause_service": y_temporal.values,
    "missing_features": missing_by_case.values
})

print(
    missing_summary
    .groupby("root_cause_service")["missing_features"]
    .agg(["count", "mean", "min", "max"])
)

                       count       mean  min  max
root_cause_service                               
adservice                 25  94.080000    7  224
cartservice               25  91.440000    0  224
checkoutservice           25  91.160000    0  224
currencyservice           24  95.958333    0  224
productcatalogservice     24  85.750000    0  224


In [ ]:
from sklearn.impute import SimpleImputer

temporal_logistic = Pipeline([
    (
        "imputer",
        SimpleImputer(
            strategy="median",
            add_indicator=True
        )
    ),
    (
        "scaler",
        StandardScaler()
    ),
    (
        "classifier",
        LogisticRegression(
            max_iter=3000,
            C=0.1,
            random_state=42
        )
    )
])

temporal_scores = cross_val_score(
    temporal_logistic,
    X_temporal,
    y_temporal,
    cv=cv,
    groups=groups,
    scoring="accuracy"
)

print("Temporal Logistic Regression fold accuracies:")

for i, score in enumerate(temporal_scores, start=1):
    print(f"Fold {i}: {score:.4f}")

print(f"\nMean accuracy: {temporal_scores.mean():.4f}")
print(f"Std deviation: {temporal_scores.std():.4f}")

Temporal Logistic Regression fold accuracies:
Fold 1: 0.5200
Fold 2: 0.6400
Fold 3: 0.4000
Fold 4: 0.6250
Fold 5: 0.7083

Mean accuracy: 0.5787
Std deviation: 0.1078


In [ ]:
from sklearn.model_selection import cross_val_predict

y_temporal_pred = cross_val_predict(
    temporal_logistic,
    X_temporal,
    y_temporal,
    cv=cv,
    groups=groups
)

print("Temporal Classification Report:\n")

print(
    classification_report(
        y_temporal,
        y_temporal_pred,
        digits=3
    )
)

temporal_cm = confusion_matrix(
    y_temporal,
    y_temporal_pred,
    labels=sorted(y_temporal.unique())
)

print("\nTemporal Confusion Matrix:")

display(
    pd.DataFrame(
        temporal_cm,
        index=sorted(y_temporal.unique()),
        columns=sorted(y_temporal.unique())
    )
)

Temporal Classification Report:

                       precision    recall  f1-score   support

            adservice      0.818     0.720     0.766        25
          cartservice      0.485     0.640     0.552        25
      checkoutservice      0.588     0.400     0.476        25
      currencyservice      0.519     0.583     0.549        24
productcatalogservice      0.542     0.542     0.542        24

             accuracy                          0.577       123
            macro avg      0.590     0.577     0.577       123
         weighted avg      0.591     0.577     0.577       123


Temporal Confusion Matrix:


,adservice,cartservice,checkoutservice,currencyservice,productcatalogservice
adservice,18,5,0,0,2
cartservice,3,16,4,0,2
checkoutservice,1,10,10,4,0
currencyservice,0,0,3,14,7
productcatalogservice,0,2,0,9,13


In [ ]:
relative_features = [
    col
    for col in X_temporal.columns
    if col.endswith("_relative_change")
]

print("Number of relative-change features:", len(relative_features))

print("\nFirst 30:")
print(relative_features[:30])

Number of relative-change features: 58

First 30:
['adservice_cpu_relative_change', 'cartservice_cpu_relative_change', 'checkoutservice_cpu_relative_change', 'currencyservice_cpu_relative_change', 'emailservice_cpu_relative_change', 'frontend_cpu_relative_change', 'main_cpu_relative_change', 'paymentservice_cpu_relative_change', 'productcatalogservice_cpu_relative_change', 'recommendationservice_cpu_relative_change', 'redis_cpu_relative_change', 'shippingservice_cpu_relative_change', 'adservice_mem_relative_change', 'cartservice_mem_relative_change', 'checkoutservice_mem_relative_change', 'currencyservice_mem_relative_change', 'emailservice_mem_relative_change', 'frontend_mem_relative_change', 'main_mem_relative_change', 'paymentservice_mem_relative_change', 'productcatalogservice_mem_relative_change', 'recommendationservice_mem_relative_change', 'redis_mem_relative_change', 'shippingservice_mem_relative_change', 'frontend-external_error_relative_change', 'frontend_error_relative_chang

In [ ]:
# Services represented in the Online Boutique telemetry
services = [
    "adservice",
    "cartservice",
    "checkoutservice",
    "currencyservice",
    "emailservice",
    "frontend",
    "main",
    "paymentservice",
    "productcatalogservice",
    "recommendationservice",
    "redis",
    "shippingservice"
]

# Map each selected metric to:
#   service + metric type
metric_groups = {}

for metric in selected_metrics:

    matched_service = None

    # Longest names first so names like productcatalogservice
    # are matched correctly.
    for service in sorted(services, key=len, reverse=True):
        prefix = service + "_"

        if metric.startswith(prefix):
            matched_service = service
            signal = metric[len(prefix):]
            break

    # Ignore metrics that are not tied to one of our services
    # (e.g. frontend-external_*).
    if matched_service is None:
        continue

    metric_groups.setdefault(signal, []).append(
        (matched_service, metric)
    )

print("Comparable telemetry groups:")

for signal, items in metric_groups.items():
    print(f"{signal:<25} -> {len(items)} services")

Comparable telemetry groups:
cpu                       -> 12 services
mem                       -> 12 services
error                     -> 2 services
workload                  -> 10 services
latency-90                -> 10 services
latency-50                -> 10 services


In [ ]:
# Only compare signals where enough services have the same telemetry type.
comparable_signals = {
    signal: items
    for signal, items in metric_groups.items()
    if len(items) >= 5
    and signal != "error"
}

print("Signals used for service-relative features:")
for signal, items in comparable_signals.items():
    print(f"{signal:<15} -> {len(items)} services")


relative_rows = []

for row_idx in range(len(X_temporal)):

    row_features = {}

    for signal, items in comparable_signals.items():

        # Collect relative-change values for this incident
        values = {}

        for service, metric in items:
            feature_name = f"{metric}_relative_change"

            if feature_name in X_temporal.columns:
                values[service] = X_temporal.iloc[
                    row_idx
                ][feature_name]

        # Remove unavailable values
        valid_values = pd.Series(values).dropna()

        if len(valid_values) == 0:
            continue

        median_change = valid_values.median()

        # Calculate each service's position relative to
        # the other services in this incident.
        for service in values:

            value = values[service]

            if pd.isna(value):
                row_features[
                    f"{service}_{signal}_median_deviation"
                ] = float("nan")

                row_features[
                    f"{service}_{signal}_rank_pct"
                ] = float("nan")

            else:
                # How far this service is from the
                # typical service for this signal.
                row_features[
                    f"{service}_{signal}_median_deviation"
                ] = value - median_change

                # Percentile rank:
                # 1.0 ≈ largest change
                # 0.0 ≈ smallest change
                rank = (
                    valid_values.rank(
                        method="average",
                        pct=True
                    )[service]
                )

                row_features[
                    f"{service}_{signal}_rank_pct"
                ] = rank

    relative_rows.append(row_features)


X_relative = pd.DataFrame(relative_rows)

print("\nRelative feature matrix shape:", X_relative.shape)
print(
    "Total missing values:",
    X_relative.isnull().sum().sum()
)

display(X_relative.head())

Signals used for service-relative features:
cpu             -> 12 services
mem             -> 12 services
workload        -> 10 services
latency-90      -> 10 services
latency-50      -> 10 services

Relative feature matrix shape: (123, 108)
Total missing values: 2940


,adservice_cpu_median_deviation,adservice_cpu_rank_pct,cartservice_cpu_median_deviation,cartservice_cpu_rank_pct,checkoutservice_cpu_median_deviation,checkoutservice_cpu_rank_pct,currencyservice_cpu_median_deviation,currencyservice_cpu_rank_pct,emailservice_cpu_median_deviation,emailservice_cpu_rank_pct,...,emailservice_latency-50_median_deviation,emailservice_latency-50_rank_pct,currencyservice_latency-50_median_deviation,currencyservice_latency-50_rank_pct,checkoutservice_latency-50_median_deviation,checkoutservice_latency-50_rank_pct,cartservice_latency-50_median_deviation,cartservice_latency-50_rank_pct,paymentservice_latency-50_median_deviation,paymentservice_latency-50_rank_pct
0,51.900481,1.0,-0.027967,0.083333,-0.000115,0.500000,0.000115,0.583333,0.002527,0.666667,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,59.161008,1.0,-0.053772,0.083333,0.007053,0.583333,-0.007053,0.500000,0.016027,0.750000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,58.564776,1.0,-0.031589,0.083333,-0.000215,0.500000,0.022700,0.916667,-0.008208,0.333333,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,34.215907,1.0,-0.022345,0.333333,-0.053998,0.083333,0.037496,0.750000,0.013972,0.583333,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,35.358672,1.0,-0.128151,0.083333,-0.008038,0.333333,0.078903,0.833333,0.007712,0.666667,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
X_v12 = pd.concat(
    [
        X_temporal.reset_index(drop=True),
        X_relative.reset_index(drop=True)
    ],
    axis=1
)

y_v12 = y_temporal.reset_index(drop=True)

print("X_v12 shape:", X_v12.shape)
print("y_v12 shape:", y_v12.shape)

print(
    "\nTotal missing values:",
    X_v12.isnull().sum().sum()
)

print("\nMissing values per case:")
print(
    X_v12.isnull().sum(axis=1)
    .value_counts()
    .sort_index()
)

X_v12 shape: (123, 514)
y_v12 shape: (123,)

Total missing values: 14218

Missing values per case:
0      37
4       2
7      28
14      7
284    49
Name: count, dtype: int64


In [ ]:
v12_logistic = Pipeline([
    (
        "imputer",
        SimpleImputer(
            strategy="median",
            add_indicator=True
        )
    ),
    (
        "scaler",
        StandardScaler()
    ),
    (
        "classifier",
        LogisticRegression(
            max_iter=4000,
            C=0.1,
            random_state=42
        )
    )
])

v12_scores = cross_val_score(
    v12_logistic,
    X_v12,
    y_v12,
    cv=cv,
    groups=groups,
    scoring="accuracy"
)

print("V1.2 Logistic Regression fold accuracies:")

for i, score in enumerate(v12_scores, start=1):
    print(f"Fold {i}: {score:.4f}")

print(f"\nMean accuracy: {v12_scores.mean():.4f}")
print(f"Std deviation: {v12_scores.std():.4f}")

V1.2 Logistic Regression fold accuracies:
Fold 1: 0.6000
Fold 2: 0.6400
Fold 3: 0.4000
Fold 4: 0.6250
Fold 5: 0.7083

Mean accuracy: 0.5947
Std deviation: 0.1037


In [ ]:
from sklearn.feature_selection import SelectKBest, mutual_info_classif

k_values = [50, 100, 150, 200, 300]

feature_selection_results = []

for k in k_values:

    model = Pipeline([
        (
            "imputer",
            SimpleImputer(
                strategy="median",
                add_indicator=True
            )
        ),
        (
            "selector",
            SelectKBest(
                score_func=mutual_info_classif,
                k=k
            )
        ),
        (
            "scaler",
            StandardScaler()
        ),
        (
            "classifier",
            LogisticRegression(
                max_iter=4000,
                C=0.1,
                random_state=42
            )
        )
    ])

    scores = cross_val_score(
        model,
        X_v12,
        y_v12,
        cv=cv,
        groups=groups,
        scoring="accuracy"
    )

    feature_selection_results.append({
        "k": k,
        "mean_accuracy": scores.mean(),
        "std": scores.std()
    })

results_df = pd.DataFrame(feature_selection_results)

display(results_df)

,k,mean_accuracy,std
0,50,0.511667,0.164452
1,100,0.454333,0.145746
2,150,0.611000,0.100806
3,200,0.667667,0.132822
4,300,0.635333,0.113771


In [ ]:
# Inspect which features are most informative

imputer = SimpleImputer(
    strategy="median",
    add_indicator=True
)

X_imputed = imputer.fit_transform(X_v12)

# Original feature names
feature_names = list(X_v12.columns)

# Get the actual columns for which missing indicators were created
missing_indicator_indices = imputer.indicator_.features_

indicator_names = [
    f"{X_v12.columns[i]}_missing"
    for i in missing_indicator_indices
]

# Complete feature-name list after imputation
all_feature_names = feature_names + indicator_names

print("Original features:", len(feature_names))
print("Missing indicators:", len(indicator_names))
print("Total imputed features:", len(all_feature_names))
print("Actual matrix columns:", X_imputed.shape[1])

selector = SelectKBest(
    score_func=mutual_info_classif,
    k=200
)

X_selected = selector.fit_transform(X_imputed, y_v12)

# Get MI scores
scores = selector.scores_

print("MI scores:", len(scores))
print("Feature names:", len(all_feature_names))

feature_scores = pd.DataFrame({
    "feature": all_feature_names,
    "mutual_information": scores
}).sort_values(
    "mutual_information",
    ascending=False
)

print("\nTop 30 informative features:")
display(feature_scores.head(30))

Original features: 514
Missing indicators: 298
Total imputed features: 812
Actual matrix columns: 812
MI scores: 812
Feature names: 812

Top 30 informative features:


,feature,mutual_information
95,cartservice_mem_late_mean,0.809042
61,productcatalogservice_cpu_delta,0.752835
24,currencyservice_cpu_middle_mean,0.730038
319,checkoutservice_latency-50_late_mean,0.729030
94,cartservice_mem_middle_mean,0.722022
320,checkoutservice_latency-50_delta,0.695470
62,productcatalogservice_cpu_relative_change,0.683048
317,checkoutservice_latency-50_early_mean,0.681839
318,checkoutservice_latency-50_middle_mean,0.677243
58,productcatalogservice_cpu_early_mean,0.669353


In [ ]:
from sklearn.svm import SVC

svm_model = Pipeline([
    (
        "imputer",
        SimpleImputer(
            strategy="median",
            add_indicator=True
        )
    ),
    (
        "selector",
        SelectKBest(
            score_func=mutual_info_classif,
            k=200
        )
    ),
    (
        "scaler",
        StandardScaler()
    ),
    (
        "classifier",
        SVC(
            kernel="rbf",
            C=1.0,
            gamma="scale"
        )
    )
])

svm_scores = cross_val_score(
    svm_model,
    X_v12,
    y_v12,
    cv=cv,
    groups=groups,
    scoring="accuracy"
)

print("Fold accuracies:")

for i, score in enumerate(svm_scores, start=1):
    print(f"Fold {i}: {score:.4f}")

print(f"\nMean accuracy: {svm_scores.mean():.4f}")
print(f"Std: {svm_scores.std():.4f}")

Fold accuracies:
Fold 1: 0.4000
Fold 2: 0.0000
Fold 3: 0.2000
Fold 4: 0.3750
Fold 5: 0.3750

Mean accuracy: 0.2700
Std: 0.1528


In [ ]:
# Analyze whether the true root-cause service tends to have
# stronger relative anomaly signals.

diagnostic_rows = []

for idx, case_name in enumerate(case_ids):

    true_root = y_temporal.iloc[idx]

    row = X_relative.iloc[idx]

    for service in services:

        scores = {}

        for signal in comparable_signals.keys():

            rank_col = f"{service}_{signal}_rank_pct"
            deviation_col = f"{service}_{signal}_median_deviation"

            if rank_col in row.index:
                scores[f"{signal}_rank"] = row[rank_col]

            if deviation_col in row.index:
                scores[f"{signal}_deviation"] = row[deviation_col]

        scores["case"] = case_name
        scores["service"] = service
        scores["is_root_cause"] = int(service == true_root)

        diagnostic_rows.append(scores)

diagnostic_df = pd.DataFrame(diagnostic_rows)

print("Rows:", len(diagnostic_df))
print("Root-cause rows:", diagnostic_df["is_root_cause"].sum())

display(
    diagnostic_df.groupby("is_root_cause")
    .mean(numeric_only=True)
)

Rows: 1476
Root-cause rows: 123


,cpu_rank,cpu_deviation,mem_rank,mem_deviation,workload_rank,workload_deviation,latency-90_rank,latency-90_deviation,latency-50_rank,latency-50_deviation
is_root_cause,,,,,,,,,,
0,0.516630,-0.003391,0.514967,0.002685,0.556907,-0.004116,0.504805,1.296169,0.507658,1.606266
1,0.817073,12.342625,0.835366,4.673454,0.487838,-0.048184,0.956757,19.828613,0.931081,52.589002


In [ ]:
# Quantify how strongly each RCA signal separates
# root-cause services from non-root-cause services.

signal_results = []

signal_columns = [
    col for col in diagnostic_df.columns
    if col.endswith("_rank") or col.endswith("_deviation")
]

for col in signal_columns:

    root_values = diagnostic_df.loc[
        diagnostic_df["is_root_cause"] == 1, col
    ].dropna()

    non_root_values = diagnostic_df.loc[
        diagnostic_df["is_root_cause"] == 0, col
    ].dropna()

    root_mean = root_values.mean()
    non_root_mean = non_root_values.mean()

    signal_results.append({
        "signal": col,
        "root_mean": root_mean,
        "non_root_mean": non_root_mean,
        "difference": root_mean - non_root_mean
    })

signal_results_df = (
    pd.DataFrame(signal_results)
    .sort_values("difference", ascending=False)
)

display(signal_results_df)

,signal,root_mean,non_root_mean,difference
9,latency-50_deviation,52.589002,1.606266,50.982736
7,latency-90_deviation,19.828613,1.296169,18.532445
1,cpu_deviation,12.342625,-0.003391,12.346016
3,mem_deviation,4.673454,0.002685,4.670769
6,latency-90_rank,0.956757,0.504805,0.451952
8,latency-50_rank,0.931081,0.507658,0.423423
2,mem_rank,0.835366,0.514967,0.320399
0,cpu_rank,0.817073,0.516630,0.300443
5,workload_deviation,-0.048184,-0.004116,-0.044067
4,workload_rank,0.487838,0.556907,-0.069069


In [ ]:
import numpy as np

rca_signals = [
    "cpu",
    "mem",
    "latency-50",
    "latency-90"
]

score_rows = []

for idx, case_name in enumerate(case_ids):

    row = X_relative.iloc[idx]

    for service in services:

        service_scores = {
            "case": case_name,
            "service": service,
            "is_root_cause": int(service == y_temporal.iloc[idx])
        }

        for signal in rca_signals:

            deviation_col = f"{service}_{signal}_median_deviation"
            rank_col = f"{service}_{signal}_rank_pct"

            deviation = row.get(deviation_col, np.nan)
            rank = row.get(rank_col, np.nan)

            service_scores[f"{signal}_deviation"] = deviation
            service_scores[f"{signal}_rank"] = rank

        score_rows.append(service_scores)

rca_scores_df = pd.DataFrame(score_rows)

print("Rows:", len(rca_scores_df))
print("Cases:", rca_scores_df["case"].nunique())
print("Services per case:",
      rca_scores_df.groupby("case")["service"].count().unique())

display(rca_scores_df.head(15))

Rows: 1476
Cases: 123
Services per case: [12]


,case,service,is_root_cause,cpu_deviation,cpu_rank,mem_deviation,mem_rank,latency-50_deviation,latency-50_rank,latency-90_deviation,latency-90_rank
0,re1ob_adservice_cpu_1,adservice,1,51.900481,1.000000,0.059236,1.000000,NaN,NaN,NaN,NaN
1,re1ob_adservice_cpu_1,cartservice,0,-0.027967,0.083333,0.002762,0.666667,NaN,NaN,NaN,NaN
2,re1ob_adservice_cpu_1,checkoutservice,0,-0.000115,0.500000,-0.000412,0.416667,NaN,NaN,NaN,NaN
3,re1ob_adservice_cpu_1,currencyservice,0,0.000115,0.583333,0.010150,0.833333,NaN,NaN,NaN,NaN
4,re1ob_adservice_cpu_1,emailservice,0,0.002527,0.666667,-0.000370,0.500000,NaN,NaN,NaN,NaN
5,re1ob_adservice_cpu_1,frontend,0,-0.007257,0.333333,-0.021965,0.083333,NaN,NaN,NaN,NaN
6,re1ob_adservice_cpu_1,main,0,-0.003725,0.416667,0.025067,0.916667,NaN,NaN,NaN,NaN
7,re1ob_adservice_cpu_1,paymentservice,0,-0.014487,0.166667,0.000370,0.583333,NaN,NaN,NaN,NaN
8,re1ob_adservice_cpu_1,productcatalogservice,0,0.022133,0.916667,-0.002259,0.250000,NaN,NaN,NaN,NaN
9,re1ob_adservice_cpu_1,recommendationservice,0,-0.012525,0.250000,-0.003637,0.166667,NaN,NaN,NaN,NaN


In [ ]:
# Simple service-level RCA scorer using available anomaly ranks

rank_columns = [
    "cpu_rank",
    "mem_rank",
    "latency-50_rank",
    "latency-90_rank"
]

rca_scores_df["rca_score"] = (
    rca_scores_df[rank_columns]
    .mean(axis=1, skipna=True)
)

# Rank services within each incident
rca_scores_df["service_rank"] = (
    rca_scores_df
    .groupby("case")["rca_score"]
    .rank(
        method="min",
        ascending=False
    )
)

# Evaluate Top-1 prediction
top1_cases = (
    rca_scores_df[rca_scores_df["service_rank"] == 1]
    .copy()
)

top1_correct = (
    top1_cases["is_root_cause"] == 1
).sum()

total_cases = rca_scores_df["case"].nunique()

top1_accuracy = top1_correct / total_cases

print(f"Top-1 correct: {top1_correct}/{total_cases}")
print(f"Top-1 RCA accuracy: {top1_accuracy:.4f}")

print("\nScore statistics:")
display(
    rca_scores_df.groupby("is_root_cause")["rca_score"]
    .agg(["mean", "median", "min", "max"])
)

print("\nExample cases:")
display(
    rca_scores_df
    .sort_values(["case", "rca_score"], ascending=[True, False])
    .groupby("case")
    .head(3)
    .head(30)
)

Top-1 correct: 94/123
Top-1 RCA accuracy: 0.7642

Score statistics:


,mean,median,min,max
is_root_cause,,,,
0,0.510803,0.500,0.083333,0.933333
1,0.895901,0.975,0.479167,1.000000



Example cases:


,case,service,is_root_cause,cpu_deviation,cpu_rank,mem_deviation,mem_rank,latency-50_deviation,latency-50_rank,latency-90_deviation,latency-90_rank,rca_score,service_rank
0,re1ob_adservice_cpu_1,adservice,1,51.900481,1.000000,0.059236,1.000000,NaN,NaN,NaN,NaN,1.000000,1.0
10,re1ob_adservice_cpu_1,redis,0,0.008466,0.833333,0.004698,0.750000,NaN,NaN,NaN,NaN,0.791667,2.0
3,re1ob_adservice_cpu_1,currencyservice,0,0.000115,0.583333,0.010150,0.833333,NaN,NaN,NaN,NaN,0.708333,3.0
12,re1ob_adservice_cpu_2,adservice,1,59.161008,1.000000,0.048691,1.000000,NaN,NaN,NaN,NaN,1.000000,1.0
22,re1ob_adservice_cpu_2,redis,0,0.025488,0.916667,0.006189,0.833333,NaN,NaN,NaN,NaN,0.875000,2.0
20,re1ob_adservice_cpu_2,productcatalogservice,0,0.019537,0.833333,0.000451,0.583333,NaN,NaN,NaN,NaN,0.708333,3.0
24,re1ob_adservice_cpu_3,adservice,1,58.564776,1.000000,0.049627,0.916667,NaN,NaN,NaN,NaN,0.958333,1.0
30,re1ob_adservice_cpu_3,main,0,0.015183,0.750000,0.081822,1.000000,NaN,NaN,NaN,NaN,0.875000,2.0
27,re1ob_adservice_cpu_3,currencyservice,0,0.022700,0.916667,0.003001,0.750000,NaN,NaN,NaN,NaN,0.833333,3.0
36,re1ob_adservice_cpu_4,adservice,1,34.215907,1.000000,0.049380,1.000000,NaN,NaN,NaN,NaN,1.000000,1.0


In [ ]:
signal_sets = {
    "CPU only": ["cpu_rank"],
    "Memory only": ["mem_rank"],
    "Latency-50 only": ["latency-50_rank"],
    "Latency-90 only": ["latency-90_rank"],
    "CPU + Memory": ["cpu_rank", "mem_rank"],
    "Latency-50 + Latency-90": [
        "latency-50_rank",
        "latency-90_rank"
    ],
    "All four": [
        "cpu_rank",
        "mem_rank",
        "latency-50_rank",
        "latency-90_rank"
    ]
}

baseline_results = []

for name, columns in signal_sets.items():

    temp = rca_scores_df.copy()

    temp["score"] = (
        temp[columns]
        .mean(axis=1, skipna=True)
    )

    temp["rank"] = (
        temp
        .groupby("case")["score"]
        .rank(
            method="min",
            ascending=False
        )
    )

    top1 = temp[temp["rank"] == 1]

    correct = top1["is_root_cause"].sum()
    total = temp["case"].nunique()

    baseline_results.append({
        "signals": name,
        "correct": int(correct),
        "total": total,
        "top1_accuracy": correct / total
    })

baseline_results_df = (
    pd.DataFrame(baseline_results)
    .sort_values(
        "top1_accuracy",
        ascending=False
    )
)

display(baseline_results_df)

,signals,correct,total,top1_accuracy
6,All four,94,123,0.764228
0,CPU only,87,123,0.707317
4,CPU + Memory,85,123,0.691057
1,Memory only,77,123,0.626016
5,Latency-50 + Latency-90,61,123,0.495935
3,Latency-90 only,58,123,0.471545
2,Latency-50 only,51,123,0.414634


In [ ]:
deviation_columns = [
    "cpu_deviation",
    "mem_deviation",
    "latency-50_deviation",
    "latency-90_deviation"
]

deviation_rank_df = rca_scores_df.copy()

for col in deviation_columns:

    rank_col = col.replace("_deviation", "_deviation_rank")

    deviation_rank_df[rank_col] = (
        deviation_rank_df
        .groupby("case")[col]
        .rank(
            method="average",
            pct=True,
            ascending=True
        )
    )

deviation_rank_columns = [
    "cpu_deviation_rank",
    "mem_deviation_rank",
    "latency-50_deviation_rank",
    "latency-90_deviation_rank"
]

deviation_rank_df["rca_score"] = (
    deviation_rank_df[deviation_rank_columns]
    .mean(axis=1, skipna=True)
)

deviation_rank_df["service_rank"] = (
    deviation_rank_df
    .groupby("case")["rca_score"]
    .rank(
        method="min",
        ascending=False
    )
)

top1 = deviation_rank_df[
    deviation_rank_df["service_rank"] == 1
]

correct = top1["is_root_cause"].sum()
total = deviation_rank_df["case"].nunique()

print(f"Top-1 correct: {correct}/{total}")
print(f"Deviation-rank RCA accuracy: {correct / total:.4f}")

print("\nSignal means:")
display(
    deviation_rank_df
    .groupby("is_root_cause")[deviation_rank_columns]
    .mean()
)

Top-1 correct: 94/123
Deviation-rank RCA accuracy: 0.7642

Signal means:


,cpu_deviation_rank,mem_deviation_rank,latency-50_deviation_rank,latency-90_deviation_rank
is_root_cause,,,,
0,0.516630,0.514967,0.507658,0.504805
1,0.817073,0.835366,0.931081,0.956757


In [ ]:
# Analyze early-vs-late anomaly behavior for RCA

onset_rows = []

temporal_signal_groups = {
    "cpu": [metric for metric in selected_metrics if metric.endswith("_cpu")],
    "mem": [metric for metric in selected_metrics if metric.endswith("_mem")],
    "latency-50": [metric for metric in selected_metrics if metric.endswith("_latency-50")],
    "latency-90": [metric for metric in selected_metrics if metric.endswith("_latency-90")]
}

for idx, case_name in enumerate(case_ids):

    true_root = y_temporal.iloc[idx]
    row = X_temporal.iloc[idx]

    for service in services:

        service_result = {
            "case": case_name,
            "service": service,
            "is_root_cause": int(service == true_root)
        }

        for signal, metrics in temporal_signal_groups.items():

            metric = f"{service}_{signal}"

            early_col = f"{metric}_early_mean"
            middle_col = f"{metric}_middle_mean"
            late_col = f"{metric}_late_mean"
            before_col = f"{metric}_before_mean"

            if early_col not in row.index:
                continue

            before = row[before_col]
            early = row[early_col]
            middle = row[middle_col]
            late = row[late_col]

            denominator = abs(before) + 1e-8

            service_result[f"{signal}_early_change"] = (
                (early - before) / denominator
            )

            service_result[f"{signal}_middle_change"] = (
                (middle - before) / denominator
            )

            service_result[f"{signal}_late_change"] = (
                (late - before) / denominator
            )

        onset_rows.append(service_result)

onset_df = pd.DataFrame(onset_rows)

print("Rows:", len(onset_df))

onset_columns = [
    col for col in onset_df.columns
    if col.endswith("_early_change")
]

display(
    onset_df
    .groupby("is_root_cause")[onset_columns]
    .mean()
)

Rows: 1476


,cpu_early_change,mem_early_change,latency-50_early_change,latency-90_early_change
is_root_cause,,,,
0,-0.040550,0.002632,1.210283,1.124763
1,11.654216,4.606685,34.032478,16.857787


In [ ]:
# Build a temporal-onset RCA score

early_change_columns = [
    "cpu_early_change",
    "mem_early_change",
    "latency-50_early_change",
    "latency-90_early_change"
]

onset_rank_df = onset_df.copy()

# Convert each early-change signal into a within-case percentile rank
for col in early_change_columns:

    rank_col = col.replace(
        "_early_change",
        "_early_rank"
    )

    onset_rank_df[rank_col] = (
        onset_rank_df
        .groupby("case")[col]
        .rank(
            method="average",
            pct=True,
            ascending=True
        )
    )

early_rank_columns = [
    "cpu_early_rank",
    "mem_early_rank",
    "latency-50_early_rank",
    "latency-90_early_rank"
]

# Average available early anomaly ranks
onset_rank_df["onset_score"] = (
    onset_rank_df[early_rank_columns]
    .mean(axis=1, skipna=True)
)

# Rank services within each incident
onset_rank_df["service_rank"] = (
    onset_rank_df
    .groupby("case")["onset_score"]
    .rank(
        method="min",
        ascending=False
    )
)

# Top-1 evaluation
top1 = onset_rank_df[
    onset_rank_df["service_rank"] == 1
]

correct = top1["is_root_cause"].sum()
total = onset_rank_df["case"].nunique()

print(f"Top-1 correct: {correct}/{total}")
print(f"Temporal-onset RCA accuracy: {correct / total:.4f}")

print("\nRoot vs non-root onset scores:")
display(
    onset_rank_df
    .groupby("is_root_cause")["onset_score"]
    .agg(["mean", "median", "min", "max"])
)

Top-1 correct: 96/123
Temporal-onset RCA accuracy: 0.7805

Root vs non-root onset scores:


,mean,median,min,max
is_root_cause,,,,
0,0.510911,0.500000,0.083333,0.958333
1,0.902846,0.979167,0.500000,1.000000


In [ ]:
# Combine abnormality and temporal-onset scores

combined_df = rca_scores_df[
    ["case", "service", "is_root_cause", "rca_score"]
].copy()

combined_df = combined_df.merge(
    onset_rank_df[
        ["case", "service", "onset_score"]
    ],
    on=["case", "service"],
    how="left"
)

# Equal-weight combination
combined_df["combined_score"] = (
    combined_df["rca_score"] +
    combined_df["onset_score"]
) / 2

# Rank services within each incident
combined_df["service_rank"] = (
    combined_df
    .groupby("case")["combined_score"]
    .rank(
        method="min",
        ascending=False
    )
)

# Top-1 evaluation
top1 = combined_df[
    combined_df["service_rank"] == 1
]

correct = top1["is_root_cause"].sum()
total = combined_df["case"].nunique()

print(f"Top-1 correct: {correct}/{total}")
print(f"Combined RCA accuracy: {correct / total:.4f}")

print("\nScore separation:")
display(
    combined_df
    .groupby("is_root_cause")[
        ["rca_score", "onset_score", "combined_score"]
    ]
    .mean()
)

Top-1 correct: 100/123
Combined RCA accuracy: 0.8130

Score separation:


,rca_score,onset_score,combined_score
is_root_cause,,,
0,0.510803,0.510911,0.510857
1,0.895901,0.902846,0.899373


In [ ]:
# Analyze the cases where the combined RCA scorer
# failed to identify the true root cause.

# Get the predicted service for each case
predictions = (
    combined_df
    .sort_values(
        ["case", "combined_score"],
        ascending=[True, False]
    )
    .groupby("case")
    .first()
    .reset_index()
)

# Get the true root-cause service
true_roots = (
    combined_df[
        combined_df["is_root_cause"] == 1
    ][["case", "service"]]
    .rename(columns={"service": "true_root"})
)

# Merge predictions with true labels
error_df = predictions.merge(
    true_roots,
    on="case"
)

error_df["correct"] = (
    error_df["service"] == error_df["true_root"]
)

# Get score of the true root cause
true_scores = (
    combined_df[
        combined_df["is_root_cause"] == 1
    ][["case", "combined_score"]]
    .rename(columns={"combined_score": "true_root_score"})
)

error_df = error_df.merge(
    true_scores,
    on="case"
)

# Difference between predicted score and true-root score
error_df["score_margin"] = (
    error_df["combined_score"]
    - error_df["true_root_score"]
)

# Add fault type
fault_info = metadata[
    ["case", "fault"]
].drop_duplicates()

error_df = error_df.merge(
    fault_info,
    on="case",
    how="left"
)

errors = (
    error_df[
        ~error_df["correct"]
    ]
    .sort_values(
        "score_margin",
        ascending=False
    )
)

print(f"Total errors: {len(errors)}")

display(
    errors[
        [
            "case",
            "fault",
            "true_root",
            "service",
            "combined_score",
            "true_root_score",
            "score_margin"
        ]
    ]
)

Total errors: 24


,case,fault,true_root,service,combined_score,true_root_score,score_margin
40,re1ob_cartservice_loss_1,loss,cartservice,checkoutservice,0.806250,0.489583,0.316667
117,re1ob_productcatalogservice_loss_5,loss,productcatalogservice,recommendationservice,0.910417,0.593750,0.316667
83,re1ob_currencyservice_delay_4,delay,currencyservice,checkoutservice,0.929167,0.625000,0.304167
116,re1ob_productcatalogservice_loss_4,loss,productcatalogservice,main,0.916667,0.616667,0.300000
30,re1ob_cartservice_delay_1,delay,cartservice,checkoutservice,0.856250,0.583333,0.272917
34,re1ob_cartservice_delay_5,delay,cartservice,checkoutservice,0.877083,0.614583,0.262500
33,re1ob_cartservice_delay_4,delay,cartservice,checkoutservice,0.825000,0.583333,0.241667
114,re1ob_productcatalogservice_loss_2,loss,productcatalogservice,recommendationservice,0.762500,0.537500,0.225000
31,re1ob_cartservice_delay_2,delay,cartservice,checkoutservice,0.835417,0.635417,0.200000
91,re1ob_currencyservice_loss_3,loss,currencyservice,main,0.833333,0.675000,0.158333


In [ ]:
# V1 evaluation: accuracy by fault type and root-cause service

# Add fault and true root information to every prediction
evaluation_df = combined_df.copy()

fault_map = metadata[
    ["case", "fault", "root_cause_service"]
].drop_duplicates()

evaluation_df = evaluation_df.merge(
    fault_map,
    on="case",
    how="left"
)

# Get the top-ranked prediction for every case
predictions = (
    evaluation_df
    .sort_values(
        ["case", "combined_score"],
        ascending=[True, False]
    )
    .groupby("case")
    .first()
    .reset_index()
)

predictions["correct"] = (
    predictions["service"] ==
    predictions["root_cause_service"]
)

# --------------------------------------------------
# Accuracy by fault type
# --------------------------------------------------

fault_results = (
    predictions
    .groupby("fault")
    .agg(
        correct=("correct", "sum"),
        total=("correct", "count")
    )
)

fault_results["accuracy"] = (
    fault_results["correct"] /
    fault_results["total"]
)

print("Accuracy by fault type:")
display(fault_results.sort_values("accuracy", ascending=False))


# --------------------------------------------------
# Accuracy by root-cause service
# --------------------------------------------------

service_results = (
    predictions
    .groupby("root_cause_service")
    .agg(
        correct=("correct", "sum"),
        total=("correct", "count")
    )
)

service_results["accuracy"] = (
    service_results["correct"] /
    service_results["total"]
)

print("\nAccuracy by root-cause service:")
display(service_results.sort_values("accuracy", ascending=False))


# --------------------------------------------------
# Overall
# --------------------------------------------------

print(
    f"\nOverall Top-1 accuracy: "
    f"{predictions['correct'].mean():.4f}"
)

Accuracy by fault type:


,correct,total,accuracy
fault,,,
cpu,24,24,1.000000
mem,25,25,1.000000
disk,24,25,0.960000
delay,15,25,0.600000
loss,11,24,0.458333



Accuracy by root-cause service:


,correct,total,accuracy
root_cause_service,,,
adservice,25,25,1.000000
checkoutservice,24,25,0.960000
productcatalogservice,17,24,0.708333
cartservice,17,25,0.680000
currencyservice,16,24,0.666667



Overall Top-1 accuracy: 0.8049


In [ ]:
from sklearn.model_selection import StratifiedGroupKFold
import numpy as np
import pandas as pd

# --------------------------------------------------
# 1. Case-level metadata
# --------------------------------------------------

case_info = (
    re1_ob[
        re1_ob["case"].isin(case_ids)
    ][
        ["case", "root_cause_service", "fault"]
    ]
    .copy()
)

case_info["group"] = (
    case_info["root_cause_service"]
    + "_"
    + case_info["fault"]
)

# Keep exactly the same case ordering
case_info = (
    case_info
    .set_index("case")
    .loc[case_ids]
    .reset_index()
)

# --------------------------------------------------
# 2. 5-fold Stratified + Grouped CV
# --------------------------------------------------

cv_strict = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

fold_results = []

for fold, (train_idx, test_idx) in enumerate(
    cv_strict.split(
        case_info,
        case_info["root_cause_service"],
        groups=case_info["group"]
    ),
    start=1
):

    # Cases belonging to this fold
    test_cases = set(
        case_info.iloc[test_idx]["case"]
    )

    # --------------------------------------------------
    # 3. Get predictions for ONLY the held-out cases
    # --------------------------------------------------

    test_predictions = (
        combined_df[
            combined_df["case"].isin(test_cases)
        ]
        .sort_values(
            ["case", "combined_score", "service"],
            ascending=[True, False, True]
        )
        .groupby("case")
        .first()
        .reset_index()
    )

    # --------------------------------------------------
    # 4. Attach actual root cause
    # --------------------------------------------------

    roots = (
        case_info[
            case_info["case"].isin(test_cases)
        ][
            ["case", "root_cause_service"]
        ]
    )

    test_predictions = test_predictions.merge(
        roots,
        on="case",
        how="left"
    )

    # --------------------------------------------------
    # 5. Check whether prediction is correct
    # --------------------------------------------------

    test_predictions["correct"] = (
        test_predictions["service"]
        == test_predictions["root_cause_service"]
    )

    # --------------------------------------------------
    # 6. Fold accuracy
    # --------------------------------------------------

    accuracy = test_predictions["correct"].mean()

    fold_results.append({
        "fold": fold,
        "test_cases": len(test_predictions),
        "correct": int(test_predictions["correct"].sum()),
        "accuracy": accuracy
    })

# --------------------------------------------------
# 7. Results
# --------------------------------------------------

fold_results_df = pd.DataFrame(fold_results)

display(fold_results_df)

mean_accuracy = fold_results_df["accuracy"].mean()
std_accuracy = fold_results_df["accuracy"].std(ddof=0)

print(f"Mean accuracy       : {mean_accuracy:.4f} ({mean_accuracy*100:.2f}%)")
print(f"Standard deviation  : {std_accuracy:.4f} ({std_accuracy*100:.2f} percentage points)")
print(f"Minimum fold        : {fold_results_df['accuracy'].min()*100:.2f}%")
print(f"Maximum fold        : {fold_results_df['accuracy'].max()*100:.2f}%")

,fold,test_cases,correct,accuracy
0,1,25,15,0.600000
1,2,25,21,0.840000
2,3,25,25,1.000000
3,4,24,18,0.750000
4,5,24,20,0.833333


Mean accuracy       : 0.8047 (80.47%)
Standard deviation  : 0.1305 (13.05 percentage points)
Minimum fold        : 60.00%
Maximum fold        : 100.00%


In [ ]:
# Recreate the exact same CV splits
cv_strict = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

fold_breakdown = []

for fold, (train_idx, test_idx) in enumerate(
    cv_strict.split(
        case_info,
        case_info["root_cause_service"],
        groups=case_info["group"]
    ),
    start=1
):

    fold_cases = case_info.iloc[test_idx].copy()

    # Get predictions for this fold
    preds = (
        combined_df[
            combined_df["case"].isin(fold_cases["case"])
        ]
        .sort_values(
            ["case", "combined_score", "service"],
            ascending=[True, False, True]
        )
        .groupby("case")
        .first()
        .reset_index()
    )

    preds = preds.merge(
        fold_cases[["case", "root_cause_service", "fault"]],
        on="case",
        how="left"
    )

    preds["correct"] = (
        preds["service"]
        == preds["root_cause_service"]
    )

    # Accuracy by fault
    fault_acc = (
        preds
        .groupby("fault")["correct"]
        .mean()
        .mul(100)
        .round(1)
    )

    row = {
        "fold": fold,
        "overall_accuracy": round(preds["correct"].mean() * 100, 1),
    }

    for fault in ["cpu", "mem", "disk", "delay", "loss"]:
        row[f"{fault}_accuracy"] = fault_acc.get(fault, np.nan)

    fold_breakdown.append(row)

fold_breakdown_df = pd.DataFrame(fold_breakdown)

display(fold_breakdown_df)

,fold,overall_accuracy,cpu_accuracy,mem_accuracy,disk_accuracy,delay_accuracy,loss_accuracy
0,1,60.0,100.0,NaN,80.0,20.0,0.0
1,2,84.0,NaN,100.0,NaN,100.0,20.0
2,3,100.0,NaN,100.0,100.0,NaN,NaN
3,4,75.0,100.0,NaN,NaN,60.0,71.4
4,5,83.3,100.0,100.0,100.0,20.0,NaN


In [ ]:
from huggingface_hub import list_repo_files

# List files/directories available in the RCAEval dataset repository
repo_files = list_repo_files(
    repo_id="phamquiluan/RCAEval",
    repo_type="dataset"
)

# Show files related to the first RE1-SS case
first_ss_case = re1_ss.iloc[0]["case"]

print("First RE1-SS case:", first_ss_case)
print("\nMatching files:\n")

matches = [
    f for f in repo_files
    if first_ss_case in f
]

for f in matches[:30]:
    print(f)

print("\nTotal matching files:", len(matches))

NameError: name 're1_ss' is not defined

In [ ]:
# Download RE1-SS cases using the actual repository structure

case_files_ss = {}

for case_name in tqdm(
    re1_ss["case"],
    desc="Downloading RE1-SS cases"
):

    metrics_path = hf_hub_download(
        repo_id="phamquiluan/RCAEval",
        filename=f"{case_name}/metrics.parquet",
        repo_type="dataset"
    )

    inject_path = hf_hub_download(
        repo_id="phamquiluan/RCAEval",
        filename=f"{case_name}/inject_time.txt",
        repo_type="dataset"
    )

    case_files_ss[case_name] = {
        "metrics": metrics_path,
        "inject_time": inject_path
    }

print("\nDownloaded:", len(case_files_ss), "cases")

In [ ]:
# Inspect the structure of one RE1-SS case

sample_case = re1_ss.iloc[0]["case"]

sample_metrics = pd.read_parquet(
    case_files_ss[sample_case]["metrics"]
)

print("Case:", sample_case)
print("Shape:", sample_metrics.shape)

print("\nColumns:")
for col in sample_metrics.columns:
    print(" -", col)

print("\nFirst 5 rows:")
display(sample_metrics.head())

In [ ]:
# Inspect RE1-SS root-cause services and fault distribution

print("Root-cause services:")
print(
    re1_ss["root_cause_service"]
    .value_counts()
    .sort_index()
)

print("\nFault types:")
print(
    re1_ss["fault"]
    .value_counts()
    .sort_index()
)

print("\nRoot cause × fault:")
display(
    pd.crosstab(
        re1_ss["root_cause_service"],
        re1_ss["fault"]
    )
)

In [ ]:
# Check whether every RE1-SS case has usable
# pre-injection and post-injection telemetry.

ss_window_results = []

for case_name in re1_ss["case"]:

    metrics = pd.read_parquet(
        case_files_ss[case_name]["metrics"]
    )

    # Read injection timestamp
    with open(
        case_files_ss[case_name]["inject_time"],
        "r"
    ) as f:
        inject_time = float(f.read().strip())

    time_values = metrics["time"].astype(float)

    before_count = (time_values < inject_time).sum()
    after_count = (time_values >= inject_time).sum()

    ss_window_results.append({
        "case": case_name,
        "inject_time": inject_time,
        "before_rows": before_count,
        "after_rows": after_count,
        "usable": (
            before_count > 0
            and after_count > 0
        )
    })

ss_windows = pd.DataFrame(ss_window_results)

print("Total cases:", len(ss_windows))
print("Usable cases:", ss_windows["usable"].sum())
print("Unusable cases:", (~ss_windows["usable"]).sum())

print("\nUnusable cases:")
display(
    ss_windows[
        ~ss_windows["usable"]
    ]
)

Total cases: 125
Usable cases: 125
Unusable cases: 0

Unusable cases:


,case,inject_time,before_rows,after_rows,usable


In [ ]:
# Extract service + signal structure dynamically from RE1-SS

sample_metrics = pd.read_parquet(
    case_files_ss[re1_ss.iloc[0]["case"]]["metrics"]
)

metric_columns = [
    c for c in sample_metrics.columns
    if c != "time"
]

# Split columns into service and signal
metric_info = []

for col in metric_columns:
    service, signal = col.rsplit("_", 1)

    metric_info.append({
        "column": col,
        "service": service,
        "signal": signal
    })

metric_info_df = pd.DataFrame(metric_info)

print("Total telemetry columns:", len(metric_info_df))

print("\nSignals:")
print(
    metric_info_df["signal"]
    .value_counts()
    .sort_index()
)

print("\nServices:")
print(
    metric_info_df["service"]
    .unique()
)

print("\nNumber of unique services:",
      metric_info_df["service"].nunique())

print("\nService × signal:")
display(
    pd.crosstab(
        metric_info_df["service"],
        metric_info_df["signal"]
    )
)

Total telemetry columns: 59

Signals:
signal
cpu           15
error          5
latency-50     8
latency-90     8
mem           15
workload       8
Name: count, dtype: int64

Services:
['carts' 'carts-db' 'catalogue' 'catalogue-db' 'front-end' 'orders'
 'orders-db' 'payment' 'queue-master' 'rabbitmq' 'rabbitmq-exporter'
 'session-db' 'shipping' 'user' 'user-db']

Number of unique services: 15

Service × signal:


signal,cpu,error,latency-50,latency-90,mem,workload
service,,,,,,
carts,1,1,1,1,1,1
carts-db,1,0,0,0,1,0
catalogue,1,0,1,1,1,1
catalogue-db,1,0,0,0,1,0
front-end,1,1,1,1,1,1
orders,1,1,1,1,1,1
orders-db,1,0,0,0,1,0
payment,1,0,1,1,1,1
queue-master,1,1,1,1,1,1


In [ ]:
# Inspect the temporal coverage around the injection point
# for one RE1-SS case.

sample_case = re1_ss.iloc[0]["case"]

metrics = pd.read_parquet(
    case_files_ss[sample_case]["metrics"]
)

with open(
    case_files_ss[sample_case]["inject_time"],
    "r"
) as f:
    inject_time = float(f.read().strip())

time_values = metrics["time"].astype(float)

print("Case:", sample_case)
print("Injection time:", inject_time)

print("\nFirst timestamp:", time_values.min())
print("Last timestamp :", time_values.max())
print("Total points   :", len(time_values))

print("\nTime before injection:",
      (time_values < inject_time).sum())

print("Time after injection :",
      (time_values >= inject_time).sum())

print("\nSeconds before injection:",
      inject_time - time_values.min())

print("Seconds after injection :",
      time_values.max() - inject_time)

Case: re1ss_carts_cpu_1
Injection time: 1693068669.0

First timestamp: 1693068309.0
Last timestamp : 1693069029.0
Total points   : 721

Time before injection: 360
Time after injection : 361

Seconds before injection: 360.0
Seconds after injection : 360.0


In [ ]:
# Build temporal RCA features for RE1-SS
#
# We only score the 5 benchmark root-cause services:
# carts, catalogue, orders, payment, user

ss_root_services = sorted(
    re1_ss["root_cause_service"].unique()
)

ss_signals = [
    "cpu",
    "mem",
    "latency-50",
    "latency-90"
]

ss_onset_rows = []

for case_name in re1_ss["case"]:

    metrics = pd.read_parquet(
        case_files_ss[case_name]["metrics"]
    )

    with open(
        case_files_ss[case_name]["inject_time"],
        "r"
    ) as f:
        inject_time = float(f.read().strip())

    # ----------------------------------------------
    # Split telemetry around injection
    # ----------------------------------------------

    before = metrics[
        metrics["time"] < inject_time
    ]

    after = metrics[
        metrics["time"] >= inject_time
    ]

    # Divide post-injection period into 3 equal
    # temporal sections: early / middle / late
    n_after = len(after)

    early_end = n_after // 3
    middle_end = 2 * n_after // 3

    early = after.iloc[:early_end]
    middle = after.iloc[early_end:middle_end]
    late = after.iloc[middle_end:]

    # ----------------------------------------------
    # Calculate features for each candidate service
    # ----------------------------------------------

    for service in ss_root_services:

        row = {
            "case": case_name,
            "service": service
        }

        for signal in ss_signals:

            col = f"{service}_{signal}"

            # All of these should exist for the
            # five root-cause services.
            before_mean = before[col].mean()
            early_mean = early[col].mean()
            middle_mean = middle[col].mean()
            late_mean = late[col].mean()
            after_mean = after[col].mean()

            # Relative changes
            early_change = (
                (early_mean - before_mean)
                / (abs(before_mean) + 1e-8)
            )

            post_change = (
                (after_mean - before_mean)
                / (abs(before_mean) + 1e-8)
            )

            row[f"{signal}_before_mean"] = before_mean
            row[f"{signal}_early_mean"] = early_mean
            row[f"{signal}_middle_mean"] = middle_mean
            row[f"{signal}_late_mean"] = late_mean
            row[f"{signal}_early_change"] = early_change
            row[f"{signal}_post_change"] = post_change

        ss_onset_rows.append(row)

ss_onset_df = pd.DataFrame(ss_onset_rows)

print("Shape:", ss_onset_df.shape)
print("\nRows:", len(ss_onset_df))
print("Cases:", ss_onset_df["case"].nunique())
print("Services:", ss_onset_df["service"].nunique())

display(ss_onset_df.head())

Shape: (625, 26)

Rows: 625
Cases: 125
Services: 5


,case,service,cpu_before_mean,cpu_early_mean,cpu_middle_mean,cpu_late_mean,cpu_early_change,cpu_post_change,mem_before_mean,mem_early_mean,...,latency-50_middle_mean,latency-50_late_mean,latency-50_early_change,latency-50_post_change,latency-90_before_mean,latency-90_early_mean,latency-90_middle_mean,latency-90_late_mean,latency-90_early_change,latency-90_post_change
0,re1ss_carts_cpu_1,carts,7.698037,36.947099,49.018161,49.121115,3.799548,4.850859,2.102733e+08,2.176226e+08,...,0.021598,0.021296,0.986938,1.457403,0.020754,0.063702,0.080789,0.078731,2.069318,2.585700
1,re1ss_carts_cpu_1,catalogue,0.907335,0.915427,0.897483,0.920208,0.008918,0.004110,1.100550e+07,1.178791e+07,...,0.003104,0.003093,0.018146,0.021892,0.004648,0.004747,0.004787,0.004768,0.021269,0.025660
2,re1ss_carts_cpu_1,orders,5.733804,3.850487,3.623345,2.654164,-0.328458,-0.411560,3.272995e+08,3.283930e+08,...,0.022458,0.021415,0.261184,0.306154,0.030506,0.061898,0.077425,0.071362,1.029018,1.302196
3,re1ss_carts_cpu_1,payment,0.616865,0.616547,0.625662,0.626534,-0.000515,0.009823,9.346048e+06,1.073541e+07,...,0.001466,0.001471,0.078728,0.086201,0.004273,0.004291,0.004293,0.004294,0.004215,0.004686
4,re1ss_carts_cpu_1,user,3.151594,3.129498,3.070956,3.096307,-0.007011,-0.016715,1.600568e+07,1.694013e+07,...,0.003554,0.003498,0.032392,0.028157,0.007245,0.007734,0.007866,0.007636,0.067540,0.069020


In [ ]:
# --------------------------------------------------
# Frozen V1 RCA scorer for RE1-SS
#
# No tuning here.
# Same scoring philosophy used for RE1-OB.
# --------------------------------------------------

ss_rank_df = ss_onset_df.copy()

# --------------------------------------------------
# 1. Rank each signal within each incident
# --------------------------------------------------

rank_features = []

for signal in [
    "cpu",
    "mem",
    "latency-50",
    "latency-90"
]:

    change_col = f"{signal}_early_change"
    rank_col = f"{signal}_early_rank"

    ss_rank_df[rank_col] = (
        ss_rank_df
        .groupby("case")[change_col]
        .rank(
            method="average",
            pct=True,
            ascending=True
        )
    )

    rank_features.append(rank_col)

# --------------------------------------------------
# 2. Combined onset score
# --------------------------------------------------

ss_rank_df["onset_score"] = (
    ss_rank_df[rank_features]
    .mean(axis=1, skipna=True)
)

# --------------------------------------------------
# 3. Deterministic Top-1 prediction
#
# If scores tie, alphabetical service name is used
# as the deterministic secondary ordering.
# --------------------------------------------------

ss_predictions = (
    ss_rank_df
    .sort_values(
        ["case", "onset_score", "service"],
        ascending=[True, False, True]
    )
    .groupby("case")
    .first()
    .reset_index()
)

# --------------------------------------------------
# 4. Add ground-truth root cause
# --------------------------------------------------

ss_truth = re1_ss[
    ["case", "root_cause_service", "fault"]
].copy()

ss_predictions = ss_predictions.merge(
    ss_truth,
    on="case",
    how="left"
)

# --------------------------------------------------
# 5. Correctness
# --------------------------------------------------

ss_predictions["correct"] = (
    ss_predictions["service"]
    == ss_predictions["root_cause_service"]
)

accuracy = ss_predictions["correct"].mean()

print(f"RE1-SS V1 Top-1 Accuracy: {accuracy:.4f}")
print(f"RE1-SS V1 Top-1 Accuracy: {accuracy*100:.2f}%")

print("\nCorrect:",
      ss_predictions["correct"].sum())

print("Total:",
      len(ss_predictions))

print("\nPredictions:")
display(
    ss_predictions[
        [
            "case",
            "service",
            "root_cause_service",
            "fault",
            "onset_score",
            "correct"
        ]
    ].head(20)
)

RE1-SS V1 Top-1 Accuracy: 0.8960
RE1-SS V1 Top-1 Accuracy: 89.60%

Correct: 112
Total: 125

Predictions:


,case,service,root_cause_service,fault,onset_score,correct
0,re1ss_carts_cpu_1,carts,carts,cpu,0.85,True
1,re1ss_carts_cpu_2,carts,carts,cpu,0.90,True
2,re1ss_carts_cpu_3,carts,carts,cpu,0.95,True
3,re1ss_carts_cpu_4,carts,carts,cpu,0.90,True
4,re1ss_carts_cpu_5,carts,carts,cpu,0.95,True
5,re1ss_carts_delay_1,carts,carts,delay,1.00,True
6,re1ss_carts_delay_2,carts,carts,delay,0.80,True
7,re1ss_carts_delay_3,carts,carts,delay,0.95,True
8,re1ss_carts_delay_4,carts,carts,delay,0.85,True
9,re1ss_carts_delay_5,carts,carts,delay,0.95,True


In [ ]:
# RE1-SS accuracy by fault type

ss_fault_accuracy = (
    ss_predictions
    .groupby("fault")["correct"]
    .agg(
        correct="sum",
        total="count",
        accuracy="mean"
    )
    .reset_index()
)

ss_fault_accuracy["accuracy"] *= 100
ss_fault_accuracy["accuracy"] = (
    ss_fault_accuracy["accuracy"].round(2)
)

display(ss_fault_accuracy)

,fault,correct,total,accuracy
0,cpu,25,25,100.0
1,delay,18,25,72.0
2,disk,25,25,100.0
3,loss,20,25,80.0
4,mem,24,25,96.0


In [ ]:
# RE1-TT: load case index and download all case files

re1_tt = cases_all[
    cases_all["dataset"] == "RE1-TT"
].copy()

print("Total RE1-TT cases:", len(re1_tt))

case_files_tt = {}

for case_name in tqdm(
    re1_tt["case"],
    desc="Downloading RE1-TT cases"
):

    metrics_path = hf_hub_download(
        repo_id="phamquiluan/RCAEval",
        filename=f"{case_name}/metrics.parquet",
        repo_type="dataset"
    )

    inject_path = hf_hub_download(
        repo_id="phamquiluan/RCAEval",
        filename=f"{case_name}/inject_time.txt",
        repo_type="dataset"
    )

    case_files_tt[case_name] = {
        "metrics": metrics_path,
        "inject_time": inject_path
    }

print("\nDownloaded:", len(case_files_tt), "cases")

Total RE1-TT cases: 125


re1tt_ts-auth-service_cpu_1/metrics.parq(…): reconstructing file:   0%|          |  0.00B /  597kB            

re1tt_ts-auth-service_cpu_1/metrics.parq(…): downloading bytes:           |  0.00B            

inject_time.txt:   0%|          | 0.00/10.0 [00:00<?, ?B/s]

re1tt_ts-auth-service_cpu_2/metrics.parq(…): reconstructing file:   0%|          |  0.00B /  733kB            

...

inject_time.txt:   0%|          | 0.00/10.0 [00:00<?, ?B/s]

re1tt_ts-travel-service_mem_5/metrics.pa(…): reconstructing file:   0%|          |  0.00B /  669kB            

re1tt_ts-travel-service_mem_5/metrics.pa(…): downloading bytes:           |  0.00B            

inject_time.txt:   0%|          | 0.00/10.0 [00:00<?, ?B/s]


Downloaded: 125 cases


In [ ]:
# Inspect the structure of one RE1-TT case

sample_case_tt = re1_tt.iloc[0]["case"]

sample_metrics_tt = pd.read_parquet(
    case_files_tt[sample_case_tt]["metrics"]
)

print("Case:", sample_case_tt)
print("Shape:", sample_metrics_tt.shape)

print("\nColumns:")
for col in sample_metrics_tt.columns:
    print(" -", col)

print("\nFirst 5 rows:")
display(sample_metrics_tt.head())

Case: re1tt_ts-auth-service_cpu_1
Shape: (961, 221)

Columns:
 - time
 - ts-admin-basic-info-service_cpu
 - ts-admin-order-service_cpu
 - ts-admin-route-service_cpu
 - ts-admin-travel-service_cpu
 - ts-admin-user-service_cpu
 - ts-assurance-mongo_cpu
 - ts-assurance-service_cpu
 - ts-auth-mongo_cpu
 - ts-auth-service_cpu
 - ts-avatar-service_cpu
 - ts-basic-service_cpu
 - ts-cancel-service_cpu
 - ts-config-mongo_cpu
 - ts-config-service_cpu
 - ts-consign-mongo_cpu
 - ts-consign-price-mongo_cpu
 - ts-consign-price-service_cpu
 - ts-consign-service_cpu
 - ts-contacts-mongo_cpu
 - ts-contacts-service_cpu
 - ts-execute-service_cpu
 - ts-food-map-mongo_cpu
 - ts-food-map-service_cpu
 - ts-food-mongo_cpu
 - ts-food-service_cpu
 - ts-inside-payment-mongo_cpu
 - ts-inside-payment-service_cpu
 - ts-news-service_cpu
 - ts-notification-service_cpu
 - ts-order-mongo_cpu
 - ts-order-other-mongo_cpu
 - ts-order-other-service_cpu
 - ts-order-service_cpu
 - ts-payment-mongo_cpu
 - ts-payment-service_c

,time,ts-admin-basic-info-service_cpu,ts-admin-order-service_cpu,ts-admin-route-service_cpu,ts-admin-travel-service_cpu,ts-admin-user-service_cpu,ts-assurance-mongo_cpu,ts-assurance-service_cpu,ts-auth-mongo_cpu,ts-auth-service_cpu,...,ts-route-service_latency-90,ts-seat-service_latency-90,ts-security-service_latency-90,ts-station-service_latency-90,ts-ticketinfo-service_latency-90,ts-train-service_latency-90,ts-travel-service_latency-90,ts-travel2-service_latency-90,ts-ui-dashboard_latency-90,ts-user-service_latency-90
0,1693186938,2.272866,0.102742,0.079021,1.869745,0.122629,0.452062,1.582828,0.765608,30.97343,...,0.016938,0.100000,0.048304,0.015057,0.079284,0.022196,0.234666,0.385243,0.624359,0.023980
1,1693186939,2.272866,0.102742,0.079021,1.869745,0.104434,0.452062,1.582828,0.762900,30.97343,...,0.016938,0.100000,0.048304,0.015080,0.079284,0.022196,0.235673,0.392532,0.581250,0.023929
2,1693186940,1.799528,0.102742,0.079021,1.860233,0.102989,0.452062,1.363927,0.762900,30.97343,...,0.016719,0.100195,0.048365,0.015085,0.079476,0.022021,0.234666,0.387007,0.581250,0.023944
3,1693186941,2.268947,0.102742,0.079021,1.860233,0.105105,0.452062,1.363927,0.762900,30.97343,...,0.016743,0.099985,0.049911,0.014886,0.077979,0.022009,0.236164,0.386043,0.581250,0.023929
4,1693186942,2.268947,0.102742,0.079021,1.860233,0.107221,0.452062,1.363927,0.762900,30.97343,...,0.016477,0.102885,0.049828,0.014733,0.076926,0.022002,0.238048,0.391212,0.581250,0.023900


In [ ]:
# Inspect RE1-TT root-cause services and fault distribution

print("Root-cause services:")
print(
    re1_tt["root_cause_service"]
    .value_counts()
    .sort_index()
)

print("\nFault types:")
print(
    re1_tt["fault"]
    .value_counts()
    .sort_index()
)

print("\nRoot cause × fault:")
display(
    pd.crosstab(
        re1_tt["root_cause_service"],
        re1_tt["fault"]
    )
)

Root-cause services:
root_cause_service
ts-auth-service      25
ts-order-service     25
ts-route-service     25
ts-train-service     25
ts-travel-service    25
Name: count, dtype: int64

Fault types:
fault
cpu      25
delay    25
disk     25
loss     25
mem      25
Name: count, dtype: int64

Root cause × fault:


fault,cpu,delay,disk,loss,mem
root_cause_service,,,,,
ts-auth-service,5,5,5,5,5
ts-order-service,5,5,5,5,5
ts-route-service,5,5,5,5,5
ts-train-service,5,5,5,5,5
ts-travel-service,5,5,5,5,5


In [ ]:
# Build RE1-TT onset features for the 5 candidate root-cause services

tt_root_services = sorted(
    re1_tt["root_cause_service"].dropna().unique()
)

print("Candidate root-cause services:")
print(tt_root_services)
print("\nNumber of candidate services:", len(tt_root_services))

Candidate root-cause services:
['ts-auth-service', 'ts-order-service', 'ts-route-service', 'ts-train-service', 'ts-travel-service']

Number of candidate services: 5


In [ ]:
print("re1_tt columns:")
print(re1_tt.columns.tolist())

print("\nFirst row:")
display(re1_tt.iloc[0])

re1_tt columns:
['case', 'dataset', 'suite', 'system', 'system_name', 'root_cause_service', 'fault', 'fault_description', 'repetition', 'inject_time', 'n_metrics', 'n_timesteps', 'time_start', 'time_end', 'duration_minutes', 'normal_timesteps', 'faulty_timesteps', 'has_logs', 'n_logs', 'has_traces', 'n_traces', 'has_root_cause_file']

First row:


,250
case,re1tt_ts-auth-service_cpu_1
dataset,RE1-TT
suite,RE1
system,tt
system_name,Train Ticket
root_cause_service,ts-auth-service
fault,cpu
fault_description,CPU stress
repetition,1
inject_time,1693187418


In [ ]:
# Safely inspect relevant variable names

names = list(globals().keys())

relevant = [
    name for name in names
    if any(word in name.lower() for word in ["tt", "metric", "case", "data", "repo"])
]

print("Relevant variables:")
for name in sorted(relevant):
    print("-", name)

Relevant variables:
- case
- case_files
- case_files_ss
- case_files_tt
- case_ids
- case_info
- case_name
- case_names
- cases
- cases_all
- classification_report
- common_metrics
- data
- fault_cases
- first_ss_case
- fold_cases
- list_repo_files
- metadata
- metadata_df
- metric
- metric_columns
- metric_counter
- metric_frequency
- metric_groups
- metric_info
- metric_info_df
- metric_sets
- metric_type
- metrics
- metrics_path
- missing_by_case
- missing_cases
- problem_cases
- re1_tt
- repo_files
- sample_case
- sample_case_tt
- sample_metrics
- sample_metrics_tt
- selected_metric_df
- selected_metrics
- temporal_case_ids
- temporal_case_names
- test_case
- test_cases
- top1_cases
- total_cases
- tt_root_services
- tt_rows
- valid_cases


In [ ]:
print("case_files_tt type:", type(case_files_tt))

if isinstance(case_files_tt, dict):
    print("Number of entries:", len(case_files_tt))
    print("\nFirst 3 entries:")
    for k, v in list(case_files_tt.items())[:3]:
        print(k, "->", v)
else:
    print("case_files_tt:")
    print(case_files_tt)

print("\n" + "=" * 60)

print("sample_metrics_tt type:", type(sample_metrics_tt))
print("sample_metrics_tt shape:", sample_metrics_tt.shape)

print("\nColumns:")
print(sample_metrics_tt.columns.tolist()[:10])

case_files_tt type: <class 'dict'>
Number of entries: 125

First 3 entries:
re1tt_ts-auth-service_cpu_1 -> {'metrics': '/root/.cache/huggingface/hub/datasets--phamquiluan--RCAEval/snapshots/afeacb11bcc94dadfd1c8f483ee4377b2b8b614e/re1tt_ts-auth-service_cpu_1/metrics.parquet', 'inject_time': '/root/.cache/huggingface/hub/datasets--phamquiluan--RCAEval/snapshots/afeacb11bcc94dadfd1c8f483ee4377b2b8b614e/re1tt_ts-auth-service_cpu_1/inject_time.txt'}
re1tt_ts-auth-service_cpu_2 -> {'metrics': '/root/.cache/huggingface/hub/datasets--phamquiluan--RCAEval/snapshots/afeacb11bcc94dadfd1c8f483ee4377b2b8b614e/re1tt_ts-auth-service_cpu_2/metrics.parquet', 'inject_time': '/root/.cache/huggingface/hub/datasets--phamquiluan--RCAEval/snapshots/afeacb11bcc94dadfd1c8f483ee4377b2b8b614e/re1tt_ts-auth-service_cpu_2/inject_time.txt'}
re1tt_ts-auth-service_cpu_3 -> {'metrics': '/root/.cache/huggingface/hub/datasets--phamquiluan--RCAEval/snapshots/afeacb11bcc94dadfd1c8f483ee4377b2b8b614e/re1tt_ts-auth-service